# Coordinated Radar Research — 02: Causal replay and display

Run all cells using a CPU runtime. The first run downloads the official ~4.17 GB
mini archive. Existing data under /content/radar-project are reused.

This is a **development baseline**, not a trained GNN or validated attack defense.
The generated offline HTML compares clean, attacked and protected runs, including
false alerts and missed detections. No fabricated results or target percentages.


In [ ]:
from pathlib import Path
import sys, subprocess, json, shutil, platform
PROJECT = Path('/content/radar-project') if Path('/content').exists() else Path.cwd() / 'colab_run'
PROJECT.mkdir(parents=True, exist_ok=True)
print('Host:', platform.platform(), sys.version)
print('Free space (GB):', round(shutil.disk_usage(PROJECT).free / 1e9, 1))
if shutil.disk_usage(PROJECT).free < 8e9:
    raise RuntimeError('At least 8 GB free space is required for this audit.')


In [ ]:
FILES = {'requirements.txt': '# Python 3.12; CPU-only audit and replay baseline. No torch/GPU needed yet.\nnuscenes-devkit==1.2.0\nnumpy==1.26.4\npandas==2.2.3\nmatplotlib==3.9.4\npytest==8.3.5\nscipy==1.17.1\nscikit-learn==1.9.1\n', 'HANDOFF.md': "# Project handoff\n\nUpdated **9 November 2026 (Asia/Calcutta)**. Delivery target: **1 December 2026**.\nAuthoritative scope: [Research_Plan.md](Research_Plan.md).\n\n## Status\n\nImplemented: data preparation, overlap audit, causal clean/attacked replay,\nresidual/CUSUM baseline, heuristic trust and conservative fusion, offline display,\nmetrics, reproducibility checks, and self-contained Colab notebook.\n\n**The full research project is not complete. No GRU/GNN has been trained.** The\ninitial baseline has many false alerts and low recall; it is not a validated defense.\n\n## Entry points\n\n| File | Purpose |\n|---|---|\n| `scripts/replay_core.py` | Annotation-free local Kalman trackers, detector, trust and CI fusion |\n| `scripts/run_replay.py` | Grouped splits, extraction, offline injection, four runs and evaluation |\n| `scripts/infer_replay.py` | Standalone inference from canonical reports only |\n| `scripts/verify_replay.py` | Exact inference reproduction without annotations/labels |\n| `scripts/render_replay.py` | Offline HTML, measured report and comparison figure |\n| `scripts/build_notebook.py` | Rebuild both portable notebooks from current source |\n| `data_manifests/mini_splits.json` | Frozen recording-grouped pilot splits |\n| `notebooks/02_causal_radar_replay.ipynb` | Colab setup, preparation, audit, replay and results ZIP |\n| `outputs/replay_pilot/demo.html` | Clean / attacked / protected playback display |\n| `outputs/replay_pilot/REPORT.md` | Actual results and limitations |\n| `DEMO_GUIDE.md` | Presentation walkthrough and claim boundaries |\n\n## Run locally\n\n```powershell\n.venv\\Scripts\\python -m pytest -q --basetemp=outputs/test_tmp\n.venv\\Scripts\\python -u scripts/run_replay.py\n.venv\\Scripts\\python -u scripts/verify_replay.py\n.venv\\Scripts\\python scripts/build_notebook.py\n```\n\nOpen `outputs/replay_pilot/demo.html` in a desktop browser; it embeds its data and\nneeds no server. Keep the output folder for linked reports and figures. The first\nlocal extraction plus four runs took about 63 seconds (not a runtime guarantee).\n\nFor Colab, upload notebook 02, choose CPU, Run all, and download the results ZIP.\nThe new notebook has not yet been run in a fresh remote Colab runtime. The embedded\npipeline was exercised locally. Initial raw download is still about 4.17 GB.\n\n## Scientific boundaries\n\n1. The earlier feasibility audit examined all mini scenes. The new manifest groups\n   `log_token` recordings before replay extraction/injection. These are pilot splits,\n   not an untouched final benchmark. Replay used only a training scene.\n2. The default demo uses the longest >=4s audited training pair episode, selected\n   deterministically without inspecting detector success.\n3. Each sensor sweep is consumed once in time order, using its calibration and ego\n   pose. No future annotation velocities enter inference. Reception time is null\n   where unavailable.\n4. Default radar validity filtering and a 60m sensor-range limit precede position\n   DBSCAN (`eps=2m`, `min_samples=1`). Centroids are reflection centers, not semantic\n   object centers. Singletons/clutter remain a documented frontend weakness.\n5. `inputs/` contains annotation-free reports. `evaluation/` contains labels,\n   reference annotations and the injection manifest. `runs/` contains decisions.\n   Original recordings and the clean exported input are preserved.\n6. Injection is **after shared clustering, before tracking**, at the position-report\n   boundary. This refines the plan's generic diagram; it is not a raw-point/RF attack.\n7. The offline injector selects centroids exclusively inside the latest annotation\n   box (+0.5m), excluding ambiguous memberships. This is a target-association proxy,\n   especially imperfect for moving targets. Two streams share a displacement process.\n8. Ground-truth boxes, target-following camera and attack timing are offline display\n   aids only. Inference receives no annotation IDs, labels or reference trajectory.\n\n## Implemented baseline\n\n- Position-only constant-velocity state `[x,y,vx,vy]`, with Hungarian gated matching.\n  Velocity comes from past positions. Doppler is not incorrectly treated as a full\n  Cartesian velocity measurement. Three accepted hits confirm a track.\n- Innovation/CUSUM thresholds, process noise and centroid uncertainty are explicit\n  **pilot heuristics**, not calibrated parameters. Trust is not a probability.\n- New/unmatched reports remain unknown and monitored. Large shifts can evade\n  association and initialize new tracks. Missing evidence alone does not lower trust.\n- Protection rejects flagged updates, coasts at most 0.8s and inflates covariance\n  with smoothed trust. Expiry/reinitialization is a failure mode, not truth recovery.\n- Complete-link spatial groups with at most one track per sensor; equal-weight\n  covariance intersection avoids treating estimates as independent evidence.\n- Single-source/degraded/multi-source-unverified statuses. Multiple agreeing sensors\n  do not establish truth. Visibility and persistent fused identities are unfinished.\n- Four paired runs: clean baseline, clean protected, attacked baseline, attacked\n  protected. All use the same frontend and fusion rule.\n\n## Initial measured results\n\nTraining scene `scene-0553`, 1,348 radar sweeps; selected target is a trailer. This is\nnot evidence of moving-target generalization. A shared 4m offset alters 320 reports\nacross the two rear radars during the recorded six-second injection window.\n\n| Report-level measure | Attacked baseline | Attacked protected |\n|---|---:|---:|\n| True positives | 4 | 41 |\n| False positives | 3,088 | 2,007 |\n| False negatives | 316 | 279 |\n| Recall | 1.25% | 12.81% |\n| Precision | 0.13% | 2.00% |\n| First detected altered-report delay | 3.776s | 0.725s |\n\nThese are poor results, preserved for diagnosis. Clean protection also raises missed\ntarget-frames from 570 to 607 while reducing unmatched hypotheses. A lower protected\nGOSPA alone cannot be attributed to attack mitigation: compare each attacked branch\nto its own clean branch. One detected event is not an event-detection benchmark.\n\nEvaluation includes all eligible annotations and emitted hypotheses. Conditional\nRMSE uses 4m gated matching; GOSPA uses `(p=2,c=10m,alpha=2)`. Background clutter is\nretained. Per-report alerts are not independent attack events. Timing currently\ncovers local updates only, excluding extraction, fusion, display and end-to-end cost.\n\n## Checks and limitations\n\n- **16 tests passed**: prior audit geometry plus causal prefix invariance, duplicate\n  sweep rejection, uncertainty, stale/rejected tracks, unmatched reports, missing\n  evidence, CI correlation behavior, known-answer metrics, repeatable injection and\n  recording-grouped split immutability.\n- `verify_replay.py` reproduces exact frames/decisions in an isolated directory with\n  only inference scripts and attacked input. Timing is excluded as nondeterministic.\n  Its result is `outputs/replay_pilot/checks/inference_isolation.json`.\n- Comparison PNG rendered and visually inspected; generated JavaScript syntax\n  checked with Node. **Browser layout/playback is not visually verified**: localhost\n  was unreachable from the in-app browser, and Computer Use stopped because URL-policy\n  enforcement is unsupported for the available Windows browser.\n- The `.venv` has no pip module (created through uv). No package installation was\n  needed locally. `uv --cache-dir .uv-cache pip check --python .venv/Scripts/python.exe`\n  checked 34 packages successfully. Keep uv's cache inside the workspace on this\n  sandboxed Windows host, or use the notebook bootstrap.\n- New Colab notebook remote execution is unverified; do not claim otherwise.\n\n## Next work, in order\n\n1. **Nov 10–12: frontend and baseline diagnosis.** Inspect overlap candidates and\n   strict-box/two-return sensitivity; examine cluster merges, clutter and association\n   switches. Improve target-like tracking with causal inputs, calibrate noise and\n   thresholds using train/validation only, save the configuration and measure clean\n   false-alert behavior before adding graph learning.\n2. **Dataset gate:** mini offers 149 sustained pair episodes but only two triple\n   episodes. Expand the real subset or narrow recovery claims. Inspect moving targets\n   separately; never manufacture overlap.\n3. **Experiment matrix:** single-stream bias, coordinated gradual drift, independent\n   two-stream perturbations, benign calibration/timing stresses, then suppression\n   with predicted observation slots and visibility uncertainty.\n4. **Nov 13–16: learning.** Cache causal track features; implement/train node-only GRU\n   then compact temporal graph model. No annotation IDs in graph construction.\n   Preserve unmatched candidates and calibrate probabilities on validation only.\n5. **Fusion/evaluation:** robust-consensus comparison, visibility/ambiguity handling,\n   persistent fused IDs, velocity/identity metrics, probability calibration,\n   end-to-end latency and multiple seeds.\n6. **Nov 17–24: frozen experiments.** Larger independent scene manifest, held-out\n   attack families, graph-vs-GRU and trust-vs-no-trust ablations, scene-level paired\n   uncertainty estimates. Do not repeatedly tune on final test recordings.\n7. **Nov 25–Dec 1: delivery.** Verify browser playback, record the demo, complete\n   paper/presentation, and reproduce in a fresh environment. Apply the research\n   plan's scope-reduction dates if baseline/GNN milestones are missed.\n\nAfter each session update this document, rebuild notebooks after source edits, log\nchecks actually run, and retain measured failures. Never present the pilot, software\ncorruptions, heuristic trust or an untrained graph model as validated radar security.\n", 'DEMO_GUIDE.md': "# Presenting the radar replay prototype\n\nOpen `outputs/replay_pilot/demo.html` in a desktop browser. It needs no server or\ninternet connection. Keep the results folder together for linked evidence files.\nThe HTML still needs a human browser check; automated visual verification was blocked.\n\n## Five-minute walkthrough\n\n1. **Problem (30s):** two radar-report streams may be manipulated together. These\n   are recorded benign nuScenes measurements with software-injected modifications,\n   not an over-the-air hardware attack.\n2. **Clean history (45s):** press Play. Dots are clustered reflection centers,\n   circles are fused estimates, ellipses are modeled uncertainty. Reference boxes\n   are optional offline overlays.\n3. **Attack (60s):** pause in the highlighted window. The middle panel receives a\n   shared 4m shift in selected rear-radar reports. The right panel receives exactly\n   the same modified input with baseline protection. Scrub to compare the same time.\n4. **Response and limits (60s):** explain residuals, sequential anomaly scores,\n   rejection and coasting. Tracks can expire or restart. Coordinated agreement can\n   evade detection, and absent evidence is not proof of malicious activity.\n5. **Evidence (60s):** read all four table rows, including clean protected. The first\n   detector has many false alerts and low recall. Lower GOSPA alone does not establish\n   successful mitigation; protection also changes clean tracking.\n6. **Research next step (45s):** frontend calibration, more attack/benign-stress\n   cases, GRU baseline, then temporal graph learning and held-out comparison.\n\n## Accurate answers\n\n- **Live radar?** No, causal replay of real recorded radar data.\n- **GNN running?** No, the running detector is innovation/CUSUM.\n- **Does inference see the boxes?** No, reference boxes and labels are offline overlays.\n- **Reliable coordinated-attack detection?** Not yet; the measured baseline misses\n  most altered reports and has many false alarms.\n- **Recover truth when all observers agree on a plausible false target?** Not in\n  general; independent evidence is needed for otherwise indistinguishable cases.\n- **Reproducible?** Sources, inputs, parameters, hashes and decision logs are saved.\n  Isolated inference checks frames/decisions without annotations or attack labels.\n\nDo not call heuristic trust a probability, one event's delay an average, or this\ndevelopment pilot a held-out accuracy result. No hardware-safety claim is supported.\n", 'README.md': "# Coordinated radar attack research\n\nImplemented milestones: **nuScenes mini ingestion, observed radar-overlap audit,\nand causal clean/attacked replay with a residual-detector baseline and offline display**.\nThe research specification is [Research_Plan.md](Research_Plan.md); no separate `Plan.md`\nwas present when implementation began. No learned attack detector has been trained yet.\nCurrent implementation status, evidence and remaining work are in [HANDOFF.md](HANDOFF.md).\n\n## Run and display the working pilot\n\nOpen `outputs/replay_pilot/demo.html` in a desktop browser. It is self-contained;\nno server is required. Use Play or the timeline to compare clean, attacked and\nprotected tracks. Reference boxes and attack labels are explicitly offline overlays.\nThe initial measured baseline has many false alarms and low attack recall; it is\na working experimental pipeline, **not a validated defense**.\n\nTo reproduce locally after dataset preparation and the overlap audit:\n\n```powershell\n.venv\\Scripts\\python -m pytest -q --basetemp=outputs/test_tmp\n.venv\\Scripts\\python -u scripts/run_replay.py\n.venv\\Scripts\\python -u scripts/verify_replay.py\n```\n\nFor Colab, upload `notebooks/02_causal_radar_replay.ipynb` and Run all. It embeds\nthe source, prepares the data, runs the audit and replay, checks isolated inference,\nand produces a downloadable results ZIP containing the presentation.\n\n`outputs/replay_pilot/REPORT.md` and `summary.json` contain measured results.\n`inputs/` contains annotation-free clean and attacked reports; `evaluation/`\ncontains labels and references; `runs/` contains causal outputs. `provenance.json`\nrecords parameters, hashes and package versions. `data_manifests/mini_splits.json`\ngroups recordings before new features or attacks are created. These are pilot splits;\nthe ten mini scenes are not a sufficient publication benchmark.\n\nThe online entry point can run without nuScenes annotations or attack labels:\n\n```powershell\n.venv\\Scripts\\python scripts/infer_replay.py outputs/replay_pilot/inputs/attacked.json outputs/replay_pilot/standalone.json --protected\n```\n\n## Google Colab (recommended)\n\nOpen `notebooks/01_nuscenes_mini_radar_audit.ipynb` with Colab's **Upload notebook**.\nUse a **CPU** runtime and **Run all**. The notebook is self-contained; it embeds the\nsame source scripts and tests as this repository. It creates an isolated Python 3.12\nenvironment, checks dependencies, downloads the official mini archive, extracts the\nradar subset, runs the audit, and displays measured results. It does not mount Drive.\n\nLive notebook created for this project:\nhttps://colab.research.google.com/drive/10yvF-r2LUNI3LQzEJMReiqtBpwGw1HVu\n\nTemporary data is stored under `/content/radar-project/data/nuscenes`. Before ending\nthe runtime, download `/content/radar-project/mini_audit_results.zip` using the final\ncell's optional `files.download(archive)` command or the Colab Files panel.\nThe notebook itself is saved in Drive, but runtime data is not automatically persistent.\n\nThe initial archive is 4,167,696,325 bytes (~4.17 GB). It contains every modality;\nselective extraction saves disk space but not bandwidth. Reserve at least 8 GB free.\nDownload is resumable. The official source is https://www.nuscenes.org/data/v1.0-mini.tgz.\nDataset use is subject to https://www.nuscenes.org/terms-of-use. Do not commit or\nredistribute raw data. No unofficial mirrors or credentials are embedded.\n\n## Local setup\n\nUse Python **3.12**, the target recommended by the current devkit documentation.\nThis workspace has `.venv`; to recreate on Windows with Python 3.12 installed:\n\n```powershell\npy -3.12 -m venv .venv\n.venv\\Scripts\\python -m pip install -r requirements.txt\n.venv\\Scripts\\python -m pytest -q\n.venv\\Scripts\\python -u scripts/prepare_nuscenes.py\n.venv\\Scripts\\python -u scripts/audit_radar_overlap.py\n```\n\nLinux/Colab equivalents use `.venv/bin/python`. There is no GPU dependency at this\nstage. `requirements.txt` pins direct dependencies; each audit records installed\ntransitive versions in `environment_freeze.txt`.\n\n## What overlap means\n\nFor each annotated keyframe, retrieve the latest sweep from each of the five radar\nstreams **at or before** the annotation time. Transform radar points with that\nsweep's calibration and ego pose into the global frame. Use the annotated object's\noriented BEV box to count supporting returns. Do not combine historical sweeps into\none apparently simultaneous observation.\n\nPrimary settings, fixed before examining results:\n\n- Vehicle/pedestrian annotations within 60 m; bicycle racks excluded.\n- Latest causal sweep no older than 100 ms; no future radar measurements.\n- Default devkit validity filters, explicitly specified in the script.\n- At least one valid return per supporting radar in a box padded by 0.5 m per side.\n- Returns falling into multiple annotation boxes are excluded from all of them.\n- A candidate episode has the **same pair or triple** supporting the **same target**\n  for at least three consecutive keyframes, spanning at least 0.9 seconds.\n\nSensitivity: 50/100/200 ms age limits, 0/0.5 m padding, 1/2 returns per sensor, and a\nstatic-box comparison. Annotation-derived velocity back-translates the box center to\nthe sweep timestamp; it may use future annotations, is an offline association aid,\nand must NEVER become an online model feature. Missing velocities use static boxes\nand are counted. Box orientation is held fixed within the short window.\n\nThis measures **observed geometric support**, not theoretical field of view, exact\nradar point identity, independent evidence, or attack detectability. Empty support\ncan reflect occlusion, radar sparsity, or filtering. Padding can include clutter.\nKeyframe continuity does not establish uninterrupted high-frequency support.\n\n## Outputs\n\n`outputs/mini_audit/` contains:\n\n| File | Purpose |\n|---|---|\n| `REPORT.md`, `summary.json` | Measured results, method, configuration and limitations |\n| `object_coverage.csv` | Every eligible object-keyframe under every configuration |\n| `sweep_inventory.csv` | Actual selected sweep tokens, ages and valid return counts |\n| `sensitivity.csv` | How timing, padding and support requirements affect counts |\n| `per_scene.csv`, `per_category.csv`, `sensor_pairs.csv` | Disaggregated evidence |\n| `overlap_episodes.csv`, `candidate_episodes.csv` | Exact fixed-pair/triple episode IDs |\n| `coverage.png`, `example_overlap.png` | Plots derived from measured data |\n| `example_manifest.json` | Scene, object and sweep IDs behind the example |\n| `environment_freeze.txt` | Installed package versions |\n\nThe dataset preparation manifest records archive and extracted-file SHA-256 hashes.\nThe audit records metadata hashes and its source hash. Reuse checks sizes; use hashes\nwhen independently verifying a transferred dataset.\n\n## Feasibility decision\n\nTwo-radar episodes support experiments that manipulate two streams. Three-radar\nepisodes may support two manipulated streams plus a remaining observer, subject to\nmanual inspection and association checks. Neither guarantees recoverability.\n\nMini's ten scenes are a **pilot**, not a sufficient training/test benchmark. Do not\nmake a decision from raw totals alone: inspect strict-box/two-return sensitivity,\nmoving-target subsets, sustained episodes, and diversity across scenes. If third-\nobserver support is scarce, narrow the radar-only claims or evaluate a larger subset\nbefore committing to a learned defense. Never generate fake overlap to rescue scope.\n\n## Development\n\nAfter source changes, rebuild the portable notebook:\n\n```powershell\npython scripts/build_notebook.py\n```\n\nTests cover causal timestamp selection, sensor-to-global transformations, rotated\nbox geometry, ambiguous-point exclusion, fixed-sensor episode continuity and archive\npath safety. The notebook executes these tests before downloading data.\n\nPrimary references:\n- https://github.com/nutonomy/nuscenes-devkit/blob/master/docs/installation.md\n- https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/nuscenes/utils/data_classes.py\n- https://www.nuscenes.org/tutorials/nuscenes_tutorial.html\n", 'Research_Plan.md': "# Coordinated radar attack detection and trust-aware fusion\n\nResearch and delivery plan prepared 7 November 2026. Deadline: 1 December 2026. Team: two people. Compute: free Google Colab. No radar boards or laboratory access.\n\n## 1. Decision and defensible scope\n\nBuild a **replay-based research prototype for detecting coordinated manipulation of automotive radar reports and limiting its effect on multi-object tracking**. Use actual recorded radar measurements, a controlled software attack injector, a small temporal graph neural network, and conservative trust-aware fusion. Deliver an executable demonstration, reproducible experiments, and a submission-ready paper by 1 December. Peer-reviewed acceptance by that date cannot be promised.\n\nRecommended working title:\n\n> Coordinated Radar-Report Attack Detection with Temporal Graph Learning and Trust-Aware Fusion: A Reproducible Evaluation on Recorded Driving Data\n\nThe original broad title can remain the project title. The paper title should identify the actual attack surface and validation level. Automotive radar is the achievable experimental domain. Air traffic and maritime radar are motivation and architectural comparisons, not validated deployment domains.\n\nThis is a research plan, not a completed experimental study. No proposed detector has been trained, dataset replayed, or performance measured in this work. Numerical settings below are proposed experimental choices, not results. The six supplied PDFs were reviewed, including methods and relevant evaluation sections; selected pages were also visually checked. External primary research, dataset documentation, and official operational reports were searched. This is a focused literature review, not an exhaustive systematic review.\n\nThree evidence categories must remain separate:\n\n| Evidence | What it supports | What it does not support |\n|---|---|---|\n| Real recorded benign radar data | Real sensor noise, clutter, geometry, and traffic conditions | Real malicious attack measurements |\n| Controlled modifications of those reports | Measured performance against the stated data-manipulation model | Feasibility of producing every modification over the air |\n| Fully synthetic geometry experiments | Exact ground truth and controlled observability studies | Performance of operational radar hardware |\n\nRandomness is appropriate for reproducible Monte Carlo experiments, sampling, and training initialization. Fabricating final accuracy values is not. Every reported outcome must come from a saved execution, with its configuration, source data, and seed.\n\n## 2. What the supplied papers contribute\n\nAll six originals are in `Research Papers/`. Page references below refer to PDF pages, with printed pagination where useful.\n\n| Supplied paper | Evidence and method | Use in this project | Limitation for the proposed claim |\n|---|---|---|---|\n| Calatrava, Tang, Closas, *Advances in Anti-Deception Jamming Strategies for Radar Systems: A Survey*, DOI 10.1109/MAES.2025.3588477; supplied issue April 2026 | Taxonomy of prevention, detection, signal-domain mitigation, and measurement-domain mitigation. Distributed radar and AI discussion on PDF pp. 18–21 | Main taxonomy and citation trail; particularly references 34, 54, 56, 58, and 60 | A survey does not establish the effectiveness of our detector. It already discusses cooperative deception and graph-based recognition; avoid broad first-ever claims |\n| Xie et al., *A Radar System With Adaptive Waveform Selection Against Dynamic Spoofing Attacks*, DOI 10.1109/TAES.2024.3499901; April 2025 issue | Authentication/detection followed by waveform selection; simulations and hardware measurements using two IWR1843 radars; PDF pp. 7–8 distinguish the experiments | Motivation for a detect-then-mitigate pipeline and eventual waveform-level extension | Requires waveform control and RF validation. A point-cloud replay cannot reproduce or validate the waveform defense |\n| Kumuda et al., *Multitarget Detection and Tracking by Mitigating Spot Jammer Attack in 77-GHz mm-Wave Radars*, DOI 10.1109/JSEN.2022.3227012; 2023 | IWR1642 victim, AWR1642 interferer, DCA1000 capture; four CFAR variants, clustering, EKF tracking; experimental setup and results on PDF pp. 10–11 onward | Practical reference pipeline and tracking-centric evaluation | One victim radar with a spot interferer, not a colluding network. Its “GNN” means **global nearest neighbor association**, not graph neural network |\n| Bi et al., *Detection and Mitigation of Position Spoofing Attacks on Cooperative UAV Swarm Formations*, DOI 10.1109/TIFS.2023.3341398; 2024 | Geometric feasibility, semidefinite relaxation, and malicious-node identification; simulation includes collusion | Shows why multiple agreeing reports can frame an honest node; motivates geometric constraints | UAV localization assumptions and distance observations do not transfer automatically to radar target tracking; no radar hardware validation |\n| Yamasaki et al., *Detection and Mitigation of Sensor-Data Tampering Attacks for Leader–Follower CACC Systems with Multiple Inter-Vehicle Distance Sensors*, DOI 10.1109/MECATRONICS-REM67547.2025.11349406; supplied 2025 proceedings | Model-based tamper detection observer and redundant measurements; ten-vehicle simulation; conclusion leaves modeling errors for future work | Strong simple baseline idea: prediction residuals and redundant sensing | Specified platoon model and attack cases; not general multi-radar coordinated deception; results are simulated |\n| Li et al., *RADAR: Run-time Adversarial Weight Attack Detection and Accuracy Recovery* | Checksums and recovery for neural-network weight bit flips in memory | Optional model-integrity background | **RADAR is an acronym, not a radar-sensor defense.** Exclude it from the core radar-method comparison |\n\nDo not copy published percentages into a target-results table. Different threat models, sensors, and metric denominators make such comparisons misleading.\n\n## 3. Closest additional work and research gap\n\nThe following are especially important before writing a novelty claim:\n\n1. **Distributed radar under registration errors.** Zhao, Yi, and Liu use received-signal correlation while accounting for registration uncertainty. This is relevant to distinguishing deception from misalignment, but access to received signals differs from access to processed reports. [Primary article](https://www.mdpi.com/1424-8220/22/19/7216).\n2. **Collaborative false targets already have radar-specific defenses.** Yang et al. combine local filtering and consensus covariance intersection; their collaborative-deception treatment uses passive radar information. This makes an independent reference an explicit architectural resource. [Primary article](https://ietresearch.onlinelibrary.wiley.com/doi/10.1049/rsn2.12371).\n3. **Coordinated multijammer research already exists.** Sun et al. study coordinated deception-jamming power scheduling against distributed radar systems. Its existence rules out claiming coordinated radar deception is a newly discovered problem. [University repository](https://arpi.unipi.it/handle/11568/1278272).\n4. **Trust-aware fusion already exists.** MATE estimates trust distributions and uses them in fusion. Its discussion explicitly identifies coordinated false reports and sparse overlapping coverage as difficulties. It is a key conceptual and experimental baseline; use the available preprint as such and verify publication/code status before submission. [Authors’ full text, especially §9](https://arxiv.org/html/2503.04954v1).\n5. **Radar-camera attack detection already exists.** Alkanat and Pandharipande, IV 2025, evaluate detection/localization using real data with simulated spoofing. The published abstract reports F1 of 0.93 for false-positive and 0.82 for false-negative attacks. These are their results, not comparable targets for ours. [University publication record](https://research.tue.nl/en/publications/deep-sensor-fusion-for-detection-and-localization-of-automotive-r/).\n6. **Graph learning itself is not sufficient novelty.** The supplied survey cites Zhu et al., *Radar Deception Jamming Recognition Based on GCN and Transformer*, ICSIDP 2024. Retrieve the primary full text before making a fine-grained comparison. Graph-based physical-layer spoofing detection also exists in wireless communications, a different sensing problem. [Authors’ preprint](https://arxiv.org/abs/2401.08220).\n7. **An additional trust-fusion paper requires closer review.** *Cyber-Resilient Perception: Safeguarding Autonomous Vehicles With Trust-Aware Sensor Fusion*, DOI 10.1109/SR.2025.3562156, is closely related. The publisher record was found, but full methods were not accessible in this review. Do not assert a gap it might already address. [Publisher record](https://ieeexplore.ieee.org/document/10971200/).\n\n**Candidate contribution, subject to verification:** a reproducible evaluation of whether visibility-aware temporal graph features improve coordinated radar-report attack detection and downstream tracking, especially under calibration errors, limited overlap, and attacks withheld during training. Include calibrated trust, a conservative fallback, and documented failure boundaries. A well-designed benchmark and a negative result can be scientifically useful; the GNN must earn its place against simpler methods.\n\n## 4. Architecture landscape and operational uses\n\nDo not conflate three independent classifications: physical sensor arrangement, where processing occurs, and the stage at which data are fused.\n\n| Architecture | Flow and representative use | Strength | Main shortcomings under attack |\n|---|---|---|---|\n| Single monostatic radar | Co-located transmitter/receiver; local detection and tracking | Simple, low communication demand | One observation source can be deceived; little independent corroboration |\n| Multiple monostatic radars | Several independently sensing radars on a vehicle or distributed sites | Different viewpoints and coverage | Overlap may be poor; common software, timing, or pose can create shared failures |\n| Bistatic/multistatic and distributed MIMO | Separated transmit/receive geometries | Additional propagation constraints and diversity | Synchronization, registration, bandwidth, association, and waveform access requirements |\n| Centralized fusion | Measurements/features sent to one fusion processor | Global evidence and straightforward experiment logging | Fusion center is a critical dependency; input corruption can poison the global estimate |\n| Decentralized/consensus fusion | Local tracks exchanged among nodes | No sole processing node; scalable local operation | Colluding nodes, uncertain cross-correlations, network partitions, repeated information |\n| Hierarchical fusion | Local processing, then regional/global fusion | Practical bandwidth/compute balance | Local errors propagate; lost low-level evidence may be impossible to recover |\n| Heterogeneous fusion | Radar combined with camera, LiDAR, inertial or navigation observations | Different sensing mechanisms can provide independent evidence | Independence is conditional; shared GNSS/time/pose can corrupt multiple modalities together |\n| Cognitive radar | Sensing results alter waveform or resource allocation | Can change how evidence is acquired | More hardware/control complexity; not achievable through recorded point clouds alone |\n\nFusion stage is another choice: **early** fusion combines raw signals/measurements, **middle** fusion combines features, and **late** fusion combines detections or tracks. Early fusion preserves evidence but demands access and calibration; late fusion is easier to prototype but can hide the origin of errors. The proposed prototype uses local track/report features and late fusion.\n\nOperational evidence: EUROCONTROL ARTAS combines radar and other surveillance reports into an air-traffic picture, demonstrating that multi-source tracking is deployed infrastructure. Its public description does not establish that it uses our proposed GNN or that it suffered a coordinated compromise. [EUROCONTROL ARTAS](https://www.eurocontrol.int/product/artas).\n\nAutomotive multi-radar sensing is directly represented by the five-radar nuScenes recording platform and four-radar RadarScenes platform. These are concrete experimental platforms, not proof of a particular production security mechanism. [nuScenes paper](https://openaccess.thecvf.com/content_CVPR_2020/papers/Caesar_nuScenes_A_Multimodal_Dataset_for_Autonomous_Driving_CVPR_2020_paper.pdf), [RadarScenes](https://radar-scenes.com/).\n\nThe technical principle behind conservative track fusion is that different estimates can share information. Covariance intersection addresses unknown correlations when its input estimates are consistent; it does not make malicious inputs trustworthy. [Research article](https://www.sciencedirect.com/science/article/abs/pii/S0921889006001436).\n\n## 5. Incidents and experimental case studies\n\n### A. Black Sea, June 2017: GNSS spoofing affecting multiple ships\n\n**Who/where/when:** C4ADS documents reports including the merchant vessel Atria near Gelendzhik, Russia, in June 2017. **What/how:** onboard navigation indicated a false inland airport position; other vessels reported similar disruption. **Result/response:** an operational navigation integrity problem and a US maritime interference advisory. C4ADS analyzes a wider pattern associated with Russian locations and protected facilities, but this does not identify the individual operator of every event. [C4ADS report and supporting report PDF](https://c4ads.org/reports/above-us-only-stars/).\n\n**Research lesson, our inference:** many affected receivers can share one corrupted source. Agreement is not independence. This is GNSS evidence, not a verified coordinated attack on radar echoes. Do not describe the event as a radar-network compromise or claim that a GNN would have prevented it.\n\n### B. Tartu, Estonia, April–June 2024: operational disruption and concrete recovery\n\n**Who/when/where:** Finnair suspended Tartu flights from 29 April through 31 May after GPS interference disrupted approaches. **What/how:** reliance on GPS-dependent approach procedures made interference operationally consequential. **Response/result:** an alternative approach that did not require GPS was introduced; flights resumed on 2 June. The airline statements establish the disruption and recovery, not a forensic identification of the transmitter. [Finnair suspension statement](https://news.cision.com/finnair/r/finnair-suspends-flights-to-tartu-for-a-month%2Cc3970152), [resumption statement](https://news.cision.com/finnair/r/finnair-resumes-flights-to-tartu%2Cc3982230).\n\n**Research lesson, our inference:** mitigation should include a useful independent fallback and a service-degradation state. An alarm alone is incomplete. The European Commission separately reported the ICAO Assembly's attribution of regional harmful GNSS interference to Russia in October 2025; this is regional attribution, not proof of the operator of these individual flights' interference. [Official statement](https://transport.ec.europa.eu/news-events/news/eu-welcomes-un-aviation-agencys-condemnation-russia-undermining-global-aviation-safety-2025-10-03_en).\n\n### C. Baltic maritime navigation, official 2026 notice\n\nThe Swedish Maritime Administration's notice 20041(T), published 22 April 2026 and included in its 20 May bulletin, reports recurring GNSS disruption and observed interference in radar and AIS. It recommends independent position checks and awareness of onboard GNSS dependencies. **Unknown:** exact transmitter, mechanism, coordination, affected radar processing layer, and attack success statistics. This is evidence of reported system interference, not proof of synchronized manipulation of several independent radar front ends. [Official bulletin, p. 3](https://ufs.sjofartsverket.se/pdf/2026/1119EN.pdf).\n\nA separate December 2025 agency announcement described adding radar beacons at ten lighthouse sites to support navigation during GPS disruption. Treat it as an announced resilience measure, not evidence that a particular radar attack was defeated. [Swedish Maritime Administration](https://www.sjofartsverket.se/en/about-us/news-and-press/news/tried-and-tested-technology-ensures-safe-maritime-navigation-during-gps-disruptions/).\n\n### D. MadRadar, Duke University / NDSS 2024: controlled radar attack demonstration\n\n**Who/what:** Hunt, Angell, Qi, Chen, and Pajic developed a black-box FMCW radar attack framework. **How at a high level:** it estimates victim configuration and manipulates detections. **Result:** controlled physical prototype experiments demonstrate adding, removing, and moving detections. This is direct radar-security evidence, but a laboratory research demonstration rather than a documented criminal incident or deployed multi-radar compromise. [NDSS primary publication](https://www.ndss-symposium.org/ndss-paper/madradar-a-black-box-physical-layer-attack-framework-on-mmwave-automotive-fmcw-radars/).\n\n**Research lesson:** insertion, suppression, and translation are justified report-level stress categories. It does not follow that arbitrary software edits are RF-realizable. Our mitigation is a proposal to evaluate, not a demonstrated defense against MadRadar hardware.\n\n### E. Supplied hardware studies\n\nThe spot-jammer paper and adaptive-waveform paper provide physical experimental evidence for interference effects and countermeasures under their setups. Their results are valuable engineering references. They neither validate a temporal GNN nor establish a public coordinated multi-radar attack dataset.\n\n**Evidence conclusion:** this review did not establish a publicly documented, fully attributed field incident with synchronized raw data proving coordinated deception of multiple radar sensors and quantifying the subsequent mitigation. That absence in this review is not proof that such incidents never occur. Do not fill this gap with speculative military anecdotes.\n\n## 6. Dataset decision and feasibility gate\n\n### Primary: nuScenes radar subset\n\nPrefer nuScenes for the main prototype because it provides five radar streams, calibration/ego-pose information, and annotated 3D objects. Use the mini release to debug ingestion, then selected train/validation scenes for the actual experiment. Mini-only results should be described as a pilot, not a robust publication benchmark. Ground-truth boxes are evaluated at labeled keyframes; do not claim measured ground truth at every intermediate radar timestamp. [Official tutorial](https://www.nuscenes.org/tutorials/nuscenes_tutorial.html).\n\nThe first two days must check access and terms, storage, stream decoding, annotation timestamps, and **actual multi-radar coverage of the same object**. Five installed sensors do not mean five independent observations per target. Produce a coverage histogram: eligible objects seen by 1, 2, 3, or more sensors within a causal time window. Coverage measurement can use labels offline; runtime features cannot.\n\nSelect enough independent scenes to support uncertainty estimates within compute limits, provisionally aiming for approximately 100–200 scenes after the pilot. This is a capacity target, not a statistical guarantee or a selected final sample size. Save exact scene IDs and report the distribution of usable overlaps. If access or useful overlap is inadequate by 10 November, move to RadarScenes and narrow the claims immediately.\n\n### Fallback: RadarScenes\n\nRadarScenes has four sensors and sequential measurements with sensor IDs, timestamps, Doppler, RCS, mounting calibration, and odometry. Its scene unit is one sensor measurement, so simultaneous fusion requires careful temporal alignment. It has 158 sequences. [Official structure](https://radar-scenes.com/dataset/structure/).\n\nIts labels describe moving-object radar points and track identity; they are not independent, continuous, high-precision physical target trajectories. Track IDs can change after stopping or occlusion. Therefore, report recovery of recorded clean observations and point/track association results separately from any physical-position claim. Do not call a clean radar centroid exact ground truth. [Official labeling](https://radar-scenes.com/dataset/labeling/).\n\n### Optional only\n\nThe Radar Ghost Dataset provides naturally occurring multipath examples useful for benign false-alarm checks. It is not an attack dataset. [Primary paper, IROS 2021; arXiv version posted 2024](https://arxiv.org/abs/2404.01437).\n\nView-of-Delft provides one front radar alongside other modalities, so it is not the default for multi-radar collusion. Access also has eligibility restrictions. [Official dataset](https://intelligent-vehicles.org/datasets/view-of-delft/).\n\nNo hardware experiment belongs in the December deliverable. No claim of RF-domain validation, interference power measurements, or live vehicle safety should appear in the paper.\n\n## 7. Threat model and limits\n\nThe principal adversary alters up to k radar-report streams before the trusted fusion process, including synchronized insertion, removal, or displacement of selected object measurements. The experiment controls the reports of compromised sensors; it does not claim the same capabilities for an RF-only attacker. Stream identity, acquisition timestamps, calibration, ego-pose, model code, and logger are initially trusted. Separate stress tests deliberately weaken calibration/pose assumptions.\n\nDefine coordination operationally: at least two manipulated streams share a false object trajectory, displacement process, or suppression objective. Independent simultaneous noise on two sensors is a separate comparator, not automatically a coordinated deception case.\n\nPrimary detection granularity is **sensor–object–time reliability**. A sensor that reports one manipulated target may still provide useful measurements of others. Aggregate sensor-level alerts are secondary. For suppression, create predicted observation slots so a missing measurement can be assessed. Missing observations outside field of view or under uncertain visibility must not automatically lower trust.\n\nKnown limits:\n\n- With one observing sensor, a plausible false report may be indistinguishable from a genuine target.\n- With two disagreeing sensors, the system may detect conflict without identifying the honest source.\n- A coordinated false trajectory consistent with every available measurement, prediction, and independent reference cannot generally be distinguished using those same observations.\n- A healthy sensor seeing an unrelated region does not resolve ambiguity for the attacked target.\n- Shared ego-pose or time errors can mimic coordination. Adding more same-platform sensors does not remove that dependency.\n- Anomaly evidence cannot establish malicious intent; use “suspected corruption” rather than claiming attacker attribution.\n\nSparse-observability results in secure state estimation formalize related recovery limits for specified linear systems. Do not apply a simple majority formula as a guarantee for nonlinear, asynchronous multi-target radar. [Primary secure-state-estimation paper](https://arxiv.org/abs/1510.02462).\n\n## 8. Complete proposed process flow\n\n```mermaid\nflowchart TD\n    A[Recorded radar streams] --> B[Validate IDs and timestamps; transform coordinates]\n    P[Calibration and ego pose] --> B\n    B --> C[Controlled report injector: evaluation branch]\n    C --> D[Per-sensor clustering and causal local trackers]\n    D --> E[Candidate associations and visibility-aware graph]\n    H[Previous accepted state and uncertainty] --> E\n    E --> F[Small spatial GNN plus causal temporal memory]\n    F --> G[Calibrated reliability and evidence confidence]\n    E --> R[Residual and sequential consistency checks]\n    R --> G\n    G --> I[Trust update with hysteresis]\n    I --> J[Downweight or quarantine suspect reports]\n    J --> K[Conservative track fusion]\n    K --> L[Observability and ambiguity check]\n    L --> M[Accepted tracks OR degraded / ambiguous output]\n    M --> H\n    M --> N[Replay display and audit logs]\n    A --> O[Clean reference branch: offline evaluation only]\n    GT[Annotations and injection manifest] --> Q[Offline metrics]\n    O --> Q\n    M --> Q\n```\n\nAnnotations, attack labels, and the clean reference branch must never be accessible to the online detector. A useful test is to remove those files after preparing the replay input and verify identical inference output.\n\n### Stage 1: canonical event format\n\nEach report carries sensor ID, acquisition time, reception time if available, coordinate frame, range/bearing/radial velocity or transformed position, available quality fields, and validity flags. Never invent SNR or other fields absent from the dataset. Keep source hashes and preserve the untouched data.\n\nTransform to a common local frame using calibrated poses. Propagate older observations to the processing epoch with uncertainty growth; never use future measurements to improve a past estimate. Reject stale or malformed records. Shared pose uncertainty must be acknowledged in consistency tests.\n\n### Stage 2: causal local tracking\n\nUse a small, documented clustering method and a Kalman/EKF tracker with state `[x, y, vx, vy]`, covariance, age, and missed-update count. Use nearest-neighbor/Hungarian association with gates as the initial implementation. Label this association algorithm separately from the neural graph model.\n\nAll methods receive the same attacked input and the same frontend. Avoid a different frontend for the proposed method that would confound the comparison. Learn or calibrate measurement-error settings from training recordings; label any heuristic covariance and test its sensitivity. Extended-object radar returns do not equal the bounding-box center; account for this in clustering, association gates, and error interpretation.\n\n### Stage 3: graph construction\n\nUse a node for each local sensor-track report, plus missing-observation nodes where a tracked target was plausibly observable. Add temporal links along local histories and cross-sensor links between candidate observations of the same object. Edge construction uses estimates and gates, never annotation track IDs. Preserve an unmatched candidate path so a manipulated report cannot disappear from monitoring simply because it falls outside an association gate.\n\nNode features: normalized prediction residual, track covariance/age, recent changes, missed detections, timestamp age, point support, and available Doppler/RCS summaries. Edge features: relative state, time difference, geometric compatibility, visibility evidence, and uncertainty. Doppler values must be projected through each sensor's line of sight; equality of raw radial velocities is not the expected relation.\n\nA useful sensor model is `z_i(t) = h_i(x(t), pose_i(t)) + noise_i(t) + attack_i(t)` for retained observations. Insertions and deletions modify the observation set rather than only this additive term.\n\n### Stage 4: compact temporal GNN\n\nBegin with two message-passing layers and one causal GRU, with small hidden dimensions chosen on validation. GraphSAGE-style aggregation with explicit edge features is sufficient for a first version; attention is optional, not automatically a trust score. Limit candidate edges and history length for Colab feasibility. Precompute frontend features to avoid repeatedly decoding the full dataset.\n\nProduce a corruption probability per observation slot and a confidence/abstention signal. Train on known injected corruptions and clean/benign-stress examples; use no future frames. A coordination edge/class head is optional only if single-observation reliability is working. Ground-truth coalition labels belong to training/evaluation, not graph construction.\n\nA graph is justified only if relational evidence improves held-out results beyond a per-node temporal model and a pooled set model. A five-node sensor-only graph may be too coarse for target-specific attacks, which is why track-level nodes are proposed.\n\n### Stage 5: trust update\n\nCalibrate model probabilities on validation data. Estimate reliability from model output and causal consistency evidence; use the same validation process for baseline detectors. A simple bounded update is `T_t = (1-alpha) T_(t-1) + alpha q_t`, where `q_t` is calibrated reliability and `alpha` controls memory. This is an engineering smoother, not a Bayesian posterior unless a probabilistic derivation is supplied.\n\nUse faster trust reduction and slower recovery if validated; retain minimum evidence requirements and an unknown state when observations are missing. Do not use trust-weighted fused output to certify itself without qualification. Keep leave-one-sensor-out residual checks or independent local predictions, and report their limitations under shared corruption. Do not multiply correlated model and residual probabilities as though independent.\n\n### Stage 6: fusion and mitigation\n\nQuarantine clearly unreliable reports; downweight borderline ones by covariance inflation, for example `P_i_eff = P_i / max(T_i, epsilon)` for retained inputs. Tune epsilon and thresholds on validation. Low-trust streams should not accumulate unlimited weak votes.\n\nFor associated local states use covariance intersection:\n\n`P_f^-1 = sum_i w_i (P_i_eff)^-1`\n\n`x_f = P_f sum_i w_i (P_i_eff)^-1 x_i`, with `w_i >= 0` and `sum_i w_i = 1`.\n\nChoose weights by a stated uncertainty criterion over admitted candidates. CI is conservative for suitable consistent inputs; arbitrary malicious bias and data-dependent selection invalidate any blanket consistency claim. Measure empirical coverage and error. Do not count the same fused prior as independent evidence in several local tracks.\n\nIf evidence supports two incompatible clusters with no trustworthy discriminator, retain alternatives or mark the target ambiguous. If no measurements remain, coast briefly using the motion model with increasing covariance, then mark the track stale. Predictions are not reconstructed observations. Restoration requires independent evidence, not a claim that the GNN can recover information that was entirely removed.\n\n### Stage 7: logging and presentation\n\nDisplay source observations, baseline tracks, protected tracks, covariance/uncertainty, trust histories, attack interval, and alert state. Clearly label injected attacks and offline reference overlays. Keep the attack label hidden from inference even if visible to the audience. Show source scene ID, code revision, model checksum, and run ID.\n\n## 9. Attack and benign-stress experiment matrix\n\nAll test modifications occur offline or inside replay, not through RF transmission.\n\n| Family | Controlled experiment | Question |\n|---|---|---|\n| Single-stream bias | Perturb one target's reports at one sensor | Can simple baselines already solve it? |\n| Coordinated false trajectory | Project one plausible false world trajectory into several sensor report frames | Does consistent false agreement defeat the method? |\n| Slow coordinated drift | Gradually displace reports within plausible kinematics | Does temporal detection add value over instantaneous gates? |\n| Coordinated ghost | Add a matching false track to two or more eligible streams | Is false-track acceptance reduced? |\n| Suppression | Remove a selected real object's reports in several streams | Is missing evidence distinguished from invisibility, and is track continuity preserved? |\n| Intermittent corruption | Alternate corruption and clean periods | Does trust recovery allow repeated exploitation? |\n| Benign faults | Timing jitter, dropout, calibration perturbation, natural clutter | Is the system confusing faults with intentional attacks? |\n| Ambiguity boundary | Corrupt all useful observers or create balanced conflicting groups | Does the system admit uncertainty instead of displaying false certainty? |\n\nFor each, vary the fraction of **eligible observers of that target**, not just the fraction of all installed sensors. Test one, two, and more compromised streams only where coverage permits. Stratify by overlap and by attack severity relative to measured residual scales. Publish numerical units and final ranges; do not choose them after inspecting test results.\n\nGenerate coordinated displacement from one shared world-frame trajectory, then derive sensor-specific range/bearing/Doppler. Independently adding the same range offset to every radar is generally not geometrically coordinated. Keep inserted-point support and attributes within documented empirical bounds, while stating that plausibility is weaker than RF realizability.\n\nMatched comparisons must include the same number of corrupted sensors and comparable attack magnitude for coordinated and uncoordinated cases. Use separate random seeds and trajectory templates for held-out attacks. A report-level replay with old timestamps tests protocol checks; a replay with updated timestamps tests semantic consistency. Keep those claims separate.\n\n## 10. Baselines, ablations, and statistical validity\n\nMinimum full-pipeline baselines:\n\n1. Same tracker/fusion without attack protection.\n2. Prediction innovation gate plus a sequential residual detector such as CUSUM.\n3. Robust geometric consensus and hand-designed temporal trust, followed by the same fusion rule.\n4. Per-node MLP/GRU detector without graph message passing, followed by the same trust/fusion.\n5. Proposed temporal graph detector with trust/fusion.\n\nAdd a permutation-invariant pooled model if affordable. A faithful MATE reproduction is valuable only if code, assumptions, and time permit. Otherwise call a simplified adaptation “MATE-inspired” and document every change; never present it as a reproduction. An oracle that excludes injected corruptions provides a reference bound under the same remaining observations, not a deployable method.\n\nEssential ablations: remove temporal memory; remove visibility handling; replace learned trust with fixed trust; compare trust downweighting with hard rejection; remove graph edges while keeping comparable model size. Prioritize the graph-vs-GRU and trust-vs-no-trust experiments if time is tight.\n\nSplit at scene/recording level **before** generating attacks, extracting overlapping windows, or fitting normalization. Keep nearby/repeated recordings grouped where possible. No clean scene and attacked version may appear across different splits. Fit noise scales, thresholds, calibration, and hyperparameters using training/validation only. Evaluate once on a frozen test manifest.\n\nReport:\n\n- Detection precision–recall curve/AUPRC, precision/recall/F1 at a validation-selected threshold, and false alerts per clean minute.\n- Per-sensor/per-object localization errors and attack-event detection delay; count never-detected attacks explicitly rather than averaging only successes.\n- Tracking position/velocity error for matched objects, false tracks, missed tracks, identity switches, and GOSPA where appropriate. GOSPA accounts for localization and cardinality error; fix its distance/cutoff/order settings before test. [Primary metric paper](https://arxiv.org/abs/1601.05585).\n- Clean-data degradation, removal of honest measurements, confidence calibration, and fraction of time degraded/abstaining.\n- End-to-end p50/p95 processing latency and memory on the actual measured hardware. Separate feature extraction, graph creation, neural inference, and fusion; GPU inference time alone is insufficient.\n\nFor nuScenes physical-position metrics, evaluate at annotation times and describe annotation limitations. For RadarScenes, distinguish clean-reference deviation from actual physical error. Never delete unmatched targets from RMSE reporting without also reporting missed/false-target metrics.\n\nRun at least three training seeds if feasible, aiming for five for the final selected model. Compute paired confidence intervals by bootstrapping independent scenes, not highly correlated frames. Report number of scenes, objects, attack events, clean minutes, and excluded cases. Sample counts and confidence intervals must be produced by scripts, not entered manually. Use a balanced development set but report low attack-prevalence behavior as well.\n\nSuccess is a measured tradeoff: lower attacked tracking error and false-track acceptance than the strongest feasible baseline, while preserving acceptable clean behavior and compute cost. Agree practical margins after the pilot and before final testing. If the graph model does not improve the tradeoff, report that honestly and retain the stronger simpler method in the demo.\n\n## 11. Demonstration script\n\nUse three selected, identified replay scenes, chosen by a stated rule rather than only the most impressive result:\n\n1. **Normal scene:** show sensor streams, tracks, and stable trust. Demonstrate a benign missed observation without a false accusation.\n2. **Coordinated manipulation:** replay the exact same recording with two eligible streams altered. Compare baseline and protected outputs. Show actual logged detection delay and error; do not prefill an expected improvement.\n3. **Failure-boundary scene:** remove independent support or present consistent corruption across all useful observers. Show the ambiguity/degraded response and explain why truth cannot be recovered from absent evidence.\n\nProvide a recorded fallback demonstration because free Colab sessions and GPUs are not guaranteed. Run the presentation from saved outputs or local CPU replay if live inference is too slow. Label playback speed separately from measured throughput. Freeze one deterministic demo configuration and retain full raw logs.\n\n## 12. Calendar and two-person ownership\n\nThere are 24 calendar days from 7 November to 1 December. This is tight; avoid building raw RF processing, multiple full datasets, a large transformer, a custom simulator, or a complex web application.\n\n| Dates | Person A: data, tracking, evaluation | Person B: literature, detector, writing | Exit condition |\n|---|---|---|---|\n| Nov 7–9 | Dataset access, mini replay, timestamp/coordinate checks, overlap audit | Literature matrix, exact threat model, baseline specifications | At least one real replay and measured coverage; otherwise activate fallback |\n| Nov 10–12 | Local tracker, paired clean/attacked replay, residual baseline | Compact node-only model and graph feature design | Baseline errors and attacks logged; labels excluded from inference |\n| Nov 13–16 | Robust fusion, conservative fallback, metric scripts | Temporal GNN, training checkpoints, validation calibration | First end-to-end result; no performance claims yet |\n| Nov 17–20 | Freeze splits and test manifests; batch runs | Main comparisons, graph/temporal/trust ablations | Complete essential experiments; freeze model choices |\n| Nov 21–24 | Confidence intervals, latency, failure-case audits | Manuscript results, related-work comparison, limitations | Tables regenerate from raw logs |\n| Nov 25–27 | Deterministic demonstration and installation notes | Full manuscript draft and references | Reproducible package and complete draft |\n| Nov 28–30 | Fresh-environment replay; fix defects | Supervisor review, claim/citation audit, venue formatting | Submission-ready paper and presentation |\n| Dec 1 | Deliver frozen project, logs, demo recording | Deliver paper; submit only after author/supervisor checks | Completed research artifact; acceptance remains external |\n\nBoth members cross-review the other's data leakage and metric code. If no functional baseline exists by 12 November, reduce the number of attack families and keep one compact model. If no stable GNN result exists by 16 November, prioritize a rigorous baseline study and clearly label the learned component exploratory. Never substitute fabricated results for a missed milestone.\n\n## 13. Implementation and compute plan\n\nUse Python for ingestion, tracking, graph features, and metrics; PyTorch for the compact model. A graph package is optional because a small explicit edge-list message-passing implementation may reduce installation friction. Pin actual tested versions when implementation begins. Use notebooks only as entry points; keep core logic in importable modules and save configuration files.\n\nSuggested structure:\n\n```text\ndata_manifests/       # source version, hashes, scene splits; no redistributed raw data\nsrc/data/            # adapters and causal transforms\nsrc/tracking/        # common frontend and association\nsrc/attacks/         # offline controlled report transformations\nsrc/models/          # node-only and graph-temporal models\nsrc/fusion/          # trust, conservative fusion, ambiguous states\nsrc/evaluation/      # metrics and scene-level confidence intervals\nconfigs/             # frozen configurations\nexperiments/         # run manifests, checkpoints, logs\ndemo/                # simple replay viewer and saved examples\npaper/               # manuscript and generated tables/figures\n```\n\nCache compact features for a subset that fits available storage. Checkpoint to persistent storage after each epoch/run. Use CPU baselines and short GPU training batches; do not design around a guaranteed GPU model or uninterrupted long session. Quantify resource limits from the pilot instead of inventing training times or hardware prices.\n\nMinimum verification: coordinate/velocity consistency, no future-frame access, split separation, visibility handling, exact attack-label alignment, known-answer metrics on tiny examples, empty-observation behavior, and replay reproducibility. These tests directly protect scientific validity.\n\n## 14. Publication plan and claims\n\nWrite the paper while running experiments: introduction and scope, closest related work, threat model/observability, method, data and injection protocol, baselines, results with uncertainty, failure cases, limitations, and reproducibility statement.\n\nChoose a supervisor-approved radar, sensor-fusion, or intelligent-transportation venue after seeing the results. Verify the current call, deadlines, fees, review requirements, and scope on the official venue site before submission. This plan does not promise a specific acceptance or use an unverified upcoming deadline. A preprint is public dissemination, not peer-reviewed acceptance. Do not select an outlet because it promises publication within days.\n\nPermissible claim after successful experiments:\n\n> On the specified held-out driving recordings and report-level attack families, the evaluated detector/fusion pipeline changed the measured detection and tracking tradeoff relative to the stated baselines.\n\nImpermissible claims from this design alone: universal attack detection; guaranteed protection of aviation or military radar; measured RF jamming resistance; real-world coordinated attack prevention; certified safety; or a physical attack dataset when the malicious part was injected in software.\n\nThe strongest possible deliverable by the deadline is an honest, reproducible result with a working demonstration, clear improvements where observed, and clear failure limits where recovery is impossible.\n\n## 15. Immediate actions and unresolved evidence\n\nStart with dataset access and the coverage audit, not model training. Retrieve the complete nearest-neighbor literature identified above, especially the inaccessible trust-aware fusion paper and GCN/Transformer paper. Confirm institutional dataset terms before downloading or redistributing any data. Save source versions and bibliographic metadata from the original publications rather than search snippets.\n\nOutstanding questions for experimentation: how much eligible multi-radar overlap exists; how reliable local tracks are; which features survive benign calibration shifts; whether graph learning beats a GRU/robust residual baseline; and how often the system must abstain. These are research questions, not assumptions to hide.\n", 'scripts/audit_radar_overlap.py': '"""Offline annotation-assisted radar coverage audit; NOT an online detector.\n\nOnly the latest causal sweep per radar is used. Annotation box velocities may\nuse future annotations: they are offline matching aids, never model features.\nBEV box membership is a coverage proxy, not ground-truth point association.\n"""\nfrom __future__ import annotations\nimport argparse\nfrom bisect import bisect_right\nfrom collections import Counter, defaultdict\nfrom functools import lru_cache\nimport hashlib\nimport importlib.metadata\nimport itertools\nimport json\nimport platform\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom pyquaternion import Quaternion\nfrom nuscenes.nuscenes import NuScenes\nfrom nuscenes.utils.data_classes import RadarPointCloud\n\nCHANNELS = [\'RADAR_FRONT\', \'RADAR_FRONT_LEFT\', \'RADAR_FRONT_RIGHT\', \'RADAR_BACK_LEFT\', \'RADAR_BACK_RIGHT\']\n\n\ndef latest_causal(times, timestamp, max_age_us):\n    idx = bisect_right(times, timestamp) - 1\n    return idx if idx >= 0 and timestamp - times[idx] <= max_age_us else None\n\n\ndef transform_points(points, calibration, pose):\n    p = Quaternion(calibration[\'rotation\']).rotation_matrix @ points[:3]\n    p += np.asarray(calibration[\'translation\'])[:, None]\n    p = Quaternion(pose[\'rotation\']).rotation_matrix @ p\n    return p + np.asarray(pose[\'translation\'])[:, None]\n\n\ndef box_membership(points, boxes, margin):\n    """Oriented horizontal boxes; dimensions are nuScenes width,length,height."""\n    hit = np.zeros((len(boxes), points.shape[1]), dtype=bool)\n    for i, box in enumerate(boxes):\n        yaw = Quaternion(box[\'rotation\']).yaw_pitch_roll[0]\n        c, s = np.cos(yaw), np.sin(yaw)\n        delta = points[:2] - np.asarray(box[\'translation\'])[:2, None]\n        local_x = c * delta[0] + s * delta[1]\n        local_y = -s * delta[0] + c * delta[1]\n        width, length, _ = box[\'size\']\n        hit[i] = (np.abs(local_x) <= length / 2 + margin) & (np.abs(local_y) <= width / 2 + margin)\n    return hit\n\n\ndef exclusive_counts(hit):\n    """Exclude returns inside multiple boxes, rather than double-counting them."""\n    unique = hit.sum(axis=0) == 1\n    return (hit & unique).sum(axis=1), int((hit.sum(axis=0) > 1).sum())\n\n\ndef sustained_episodes(rows, threshold, max_gap_s=0.75):\n    """Consecutive keyframes for a FIXED sensor pair/triple and target."""\n    groups = defaultdict(list)\n    for row in rows:\n        for combo in itertools.combinations(row[\'sensors\'], threshold):\n            groups[(row[\'scene\'], row[\'instance_token\'], combo)].append(row)\n    episodes = []\n    for (scene, instance, combo), records in groups.items():\n        records.sort(key=lambda r: r[\'timestamp_us\'])\n        chunks = []\n        chunk = []\n        for r in records:\n            if chunk and (r[\'frame_index\'] != chunk[-1][\'frame_index\'] + 1 or (r[\'timestamp_us\'] - chunk[-1][\'timestamp_us\']) / 1e6 > max_gap_s):\n                chunks.append(chunk)\n                chunk = []\n            chunk.append(r)\n        if chunk:\n            chunks.append(chunk)\n        for chunk in chunks:\n            episodes.append({\'scene\': scene, \'instance_token\': instance, \'sensors\': \'|\'.join(combo),\n                             \'sensor_count\': threshold, \'keyframes\': len(chunk),\n                             \'duration_s\': (chunk[-1][\'timestamp_us\'] - chunk[0][\'timestamp_us\']) / 1e6,\n                             \'start_sample\': chunk[0][\'sample_token\'], \'end_sample\': chunk[-1][\'sample_token\'],\n                             \'category\': chunk[0][\'category\']})\n    return sorted(episodes, key=lambda e: (e[\'keyframes\'], e[\'duration_s\']), reverse=True)\n\n\ndef file_hash(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef run(dataroot, output, max_range=60.0):\n    started = time.monotonic()\n    root, out = Path(dataroot), Path(output)\n    out.mkdir(parents=True, exist_ok=True)\n    nusc = NuScenes(version=\'v1.0-mini\', dataroot=str(root), verbose=True)\n    # Build chronological streams from ALL sample_data, including non-keyframes.\n    streams = defaultdict(list)\n    for sd in nusc.sample_data:\n        if sd[\'channel\'] in CHANNELS:\n            scene_token = nusc.get(\'sample\', sd[\'sample_token\'])[\'scene_token\']\n            streams[(scene_token, sd[\'channel\'])].append(sd)\n    for records in streams.values():\n        records.sort(key=lambda r: r[\'timestamp\'])\n    times = {key: [r[\'timestamp\'] for r in records] for key, records in streams.items()}\n    missing = [sd[\'filename\'] for records in streams.values() for sd in records if not (root / sd[\'filename\']).is_file()]\n    if missing:\n        raise FileNotFoundError(f\'{len(missing)} radar files missing, e.g. {missing[:3]}\')\n\n    @lru_cache(maxsize=128)\n    def load(sd_token):\n        sd = nusc.get(\'sample_data\', sd_token)\n        # Explicit devkit defaults; do not silently enable all invalid returns.\n        pc = RadarPointCloud.from_file(str(root / sd[\'filename\']), invalid_states=[0], dynprop_states=list(range(7)), ambig_states=[3])\n        points = pc.points[:, np.isfinite(pc.points[:3]).all(axis=0)]\n        cs = nusc.get(\'calibrated_sensor\', sd[\'calibrated_sensor_token\'])\n        pose = nusc.get(\'ego_pose\', sd[\'ego_pose_token\'])\n        return transform_points(points, cs, pose)\n\n    configs = [{\'id\': f\'age{age}_margin{margin:g}_points{minimum}\', \'max_age_ms\': age,\n                \'margin_m\': margin, \'min_points\': minimum}\n               for age in (50, 100, 200) for margin in (0.0, 0.5) for minimum in (1, 2)]\n    # Sensitivity to using offline target velocity to align box centers.\n    configs.append({\'id\': \'age100_margin0.5_points1_static_boxes\', \'max_age_ms\': 100,\n                    \'margin_m\': 0.5, \'min_points\': 1, \'static_boxes\': True})\n    primary_id = \'age100_margin0.5_points1\'\n    rows, sweep_rows = [], []\n    best = None\n    for scene in nusc.scene:\n        token, frame = scene[\'first_sample_token\'], 0\n        while token:\n            sample = nusc.get(\'sample\', token)\n            t = sample[\'timestamp\']\n            lidar_sd = nusc.get(\'sample_data\', sample[\'data\'][\'LIDAR_TOP\'])\n            ego = nusc.get(\'ego_pose\', lidar_sd[\'ego_pose_token\'])\n            # All boxes participate in ambiguity exclusion, including out-of-scope categories.\n            anns = [nusc.get(\'sample_annotation\', a) for a in sample[\'anns\']]\n            velocities = [nusc.box_velocity(a[\'token\']) for a in anns]\n            selected = {}\n            for channel in CHANNELS:\n                key = (scene[\'token\'], channel)\n                idx = latest_causal(times.get(key, []), t, 200_000)\n                if idx is None:\n                    sweep_rows.append({\'scene\': scene[\'name\'], \'sample_token\': token, \'sensor\': channel,\n                                       \'timestamp_us\': t, \'age_ms\': None, \'point_count\': 0, \'sample_data_token\': \'\'})\n                    continue\n                sd = streams[key][idx]\n                assert sd[\'timestamp\'] <= t\n                age = (t - sd[\'timestamp\']) / 1000\n                points = load(sd[\'token\'])\n                counts = {}\n                for static in (False, True):\n                    shifted = []\n                    for a, v in zip(anns, velocities):\n                        center = np.asarray(a[\'translation\'], dtype=float).copy()\n                        if not static and np.isfinite(v[:2]).all():\n                            center[:2] -= v[:2] * age / 1000\n                        shifted.append({**a, \'translation\': center.tolist()})\n                    for margin in (0.0, 0.5):\n                        counts[(margin, static)] = exclusive_counts(box_membership(points, shifted, margin))\n                selected[channel] = (sd, age, points, counts)\n                sweep_rows.append({\'scene\': scene[\'name\'], \'sample_token\': token, \'sensor\': channel,\n                                   \'timestamp_us\': t, \'age_ms\': age, \'point_count\': points.shape[1],\n                                   \'sample_data_token\': sd[\'token\'],\n                                   \'ambiguous_points_margin0.5\': counts[(0.5, False)][1]})\n            for j, (ann, velocity) in enumerate(zip(anns, velocities)):\n                category = ann[\'category_name\']\n                eligible_class = (category.startswith(\'vehicle.\') and category != \'vehicle.bicycle_rack\') or category.startswith(\'human.pedestrian.\')\n                distance = float(np.linalg.norm(np.asarray(ann[\'translation\'])[:2] - np.asarray(ego[\'translation\'])[:2]))\n                if not eligible_class or distance > max_range:\n                    continue\n                speed = float(np.linalg.norm(velocity[:2])) if np.isfinite(velocity[:2]).all() else None\n                for config in configs:\n                    support = {}\n                    for channel, (_, age, _, counts) in selected.items():\n                        support[channel] = int(counts[(config[\'margin_m\'], config.get(\'static_boxes\', False))][0][j]) if age <= config[\'max_age_ms\'] else 0\n                    sensors = [channel for channel in CHANNELS if support.get(channel, 0) >= config[\'min_points\']]\n                    row = {\'config\': config[\'id\'], \'scene\': scene[\'name\'], \'scene_token\': scene[\'token\'],\n                           \'sample_token\': token, \'timestamp_us\': t, \'frame_index\': frame,\n                           \'annotation_token\': ann[\'token\'], \'instance_token\': ann[\'instance_token\'],\n                           \'category\': category, \'range_m\': distance, \'speed_mps\': speed,\n                           \'velocity_available\': speed is not None, \'sensor_count\': len(sensors), \'sensors\': sensors,\n                           **{f\'points_{c}\': support.get(c, 0) for c in CHANNELS}}\n                    rows.append(row)\n                    if config[\'id\'] == primary_id and (best is None or len(sensors) > best[\'count\']):\n                        best = {\'count\': len(sensors), \'sample\': sample, \'ann\': ann, \'selected\': selected,\n                                \'ego\': ego, \'scene\': scene[\'name\'], \'row\': row}\n            token, frame = sample[\'next\'], frame + 1\n        print(f\'Audited {scene["name"]}: {frame} keyframes\', flush=True)\n\n    df = pd.DataFrame(rows)\n    serial = df.copy()\n    serial[\'sensors\'] = serial[\'sensors\'].apply(lambda x: \'|\'.join(x))\n    serial.to_csv(out / \'object_coverage.csv\', index=False)\n    pd.DataFrame(sweep_rows).to_csv(out / \'sweep_inventory.csv\', index=False)\n    summaries = []\n    for config in configs:\n        subset = df[df.config == config[\'id\']]\n        hist = subset.sensor_count.value_counts().to_dict()\n        summaries.append({**config, \'object_keyframes\': len(subset),\n                          **{f\'sensors_{i}\': int(hist.get(i, 0)) for i in range(6)},\n                          \'at_least_2\': int((subset.sensor_count >= 2).sum()),\n                          \'at_least_3\': int((subset.sensor_count >= 3).sum())})\n    pd.DataFrame(summaries).to_csv(out / \'sensitivity.csv\', index=False)\n    primary = df[df.config == primary_id]\n    episodes = sustained_episodes(primary.to_dict(\'records\'), 2) + sustained_episodes(primary.to_dict(\'records\'), 3)\n    episode_cols = [\'scene\',\'instance_token\',\'sensors\',\'sensor_count\',\'keyframes\',\'duration_s\',\'start_sample\',\'end_sample\',\'category\']\n    ep = pd.DataFrame(episodes, columns=episode_cols)\n    ep.to_csv(out / \'overlap_episodes.csv\', index=False)\n    candidates = ep[(ep.keyframes >= 3) & (ep.duration_s >= 0.9)]\n    candidates.to_csv(out / \'candidate_episodes.csv\', index=False)\n    for grouping, filename in [(\'scene\', \'per_scene.csv\'), (\'category\', \'per_category.csv\')]:\n        group_rows = []\n        for label, g in primary.groupby(grouping):\n            group_rows.append({grouping: label, \'object_keyframes\': len(g), \'observed\': int((g.sensor_count >= 1).sum()),\n                               \'at_least_2\': int((g.sensor_count >= 2).sum()), \'at_least_3\': int((g.sensor_count >= 3).sum()),\n                               \'unique_instances_2plus\': g[g.sensor_count >= 2].instance_token.nunique()})\n        pd.DataFrame(group_rows).to_csv(out / filename, index=False)\n    pair_counts = Counter()\n    for sensors in primary.sensors:\n        pair_counts.update(itertools.combinations(sensors, 2))\n    pd.DataFrame([{\'sensor_a\': a, \'sensor_b\': b, \'object_keyframes\': n} for (a,b),n in pair_counts.items()]).to_csv(out / \'sensor_pairs.csv\', index=False)\n    moving = primary[primary.speed_mps >= 0.5]\n    manifest = {\'dataset\': \'nuScenes v1.0-mini\', \'scenes\': len(nusc.scene), \'keyframes\': len(nusc.sample),\n                \'radar_files\': sum(map(len, streams.values())), \'primary\': next(s for s in summaries if s[\'id\'] == primary_id),\n                \'unique_objects_with_2plus\': int(primary[primary.sensor_count >= 2].instance_token.nunique()),\n                \'unique_objects_with_3plus\': int(primary[primary.sensor_count >= 3].instance_token.nunique()),\n                \'candidate_pair_episodes\': int((candidates.sensor_count == 2).sum()),\n                \'candidate_triple_episodes\': int((candidates.sensor_count == 3).sum()),\n                \'candidate_pair_scenes\': int(candidates[candidates.sensor_count == 2].scene.nunique()),\n                \'candidate_triple_scenes\': int(candidates[candidates.sensor_count == 3].scene.nunique()),\n                \'moving_object_keyframes\': len(moving), \'moving_2plus\': int((moving.sensor_count >= 2).sum()),\n                \'moving_3plus\': int((moving.sensor_count >= 3).sum()),\n                \'unknown_velocity_object_keyframes\': int(primary.speed_mps.isna().sum()),\n                \'parameters\': {\'max_range_m\': max_range, \'minimum_episode_keyframes\': 3, \'minimum_episode_duration_s\': 0.9,\n                               \'radar_filters\': {\'invalid_states\': [0], \'dynprop_states\': list(range(7)), \'ambig_states\': [3]},\n                               \'matching\': \'BEV oriented box, exclusive points, latest causal sweep per sensor\',\n                               \'target_alignment\': \'offline annotation velocity backward translation; static fallback if unavailable\'},\n                \'elapsed_s\': time.monotonic() - started, \'python\': sys.version, \'platform\': platform.platform(),\n                \'packages\': {p: importlib.metadata.version(p) for p in [\'nuscenes-devkit\',\'numpy\',\'pandas\',\'matplotlib\',\'pyquaternion\']},\n                \'metadata_sha256\': {p.name: file_hash(p) for p in sorted((root/\'v1.0-mini\').glob(\'*.json\'))},\n                \'code_sha256\': file_hash(__file__), \'scene_tokens\': [s[\'token\'] for s in nusc.scene],\n                \'limitations\': [\'Offline annotation-assisted coverage proxy, not a detector or exact point identity.\',\n                                \'Mini is a feasibility pilot, not training or security validation.\',\n                                \'Velocity can use future annotations for offline matching only.\',\n                                \'BEV padding can overcount clutter; strict/two-point sensitivities are provided.\',\n                                \'Observed overlap is not a guarantee of independence, visibility, or recoverability.\',\n                                \'Episodes are keyframe observations; continuity between keyframes is not established.\']}\n    prep = root / \'preparation_manifest.json\'\n    if prep.exists():\n        manifest[\'archive\'] = json.loads(prep.read_text(encoding=\'utf-8\'))[\'archive\']\n    (out / \'summary.json\').write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    with (out / \'environment_freeze.txt\').open(\'w\', encoding=\'utf-8\') as f:\n        # importlib avoids reliance on pip being installed in a uv-created venv.\n        f.write(\'\\n\'.join(sorted(f\'{d.metadata["Name"]}=={d.version}\' for d in importlib.metadata.distributions())))\n    plots(primary, pd.DataFrame(summaries), best, out)\n    write_report(manifest, summaries, candidates, out)\n    print(json.dumps({k:v for k,v in manifest.items() if k in (\'primary\',\'candidate_pair_episodes\',\'candidate_triple_episodes\',\'moving_2plus\',\'moving_3plus\')}, indent=2))\n    return manifest\n\n\ndef plots(primary, sensitivity, best, out):\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n    counts = primary.sensor_count.value_counts().reindex(range(6), fill_value=0)\n    axes[0].bar(range(6), counts.values, color=\'#277da1\')\n    axes[0].set(xlabel=\'Distinct supporting radars\', ylabel=\'Annotated object-keyframes\', title=\'Observed radar support (primary settings)\', xticks=range(6))\n    for x,y in enumerate(counts.values):\n        axes[0].text(x,y,str(y),ha=\'center\',va=\'bottom\',fontsize=9)\n    subset = sensitivity[(sensitivity.max_age_ms == 100) & ~sensitivity.id.str.contains(\'static\')]\n    x = np.arange(len(subset))\n    axes[1].bar(x-.18,subset.at_least_2,width=.36,label=\'2+ radars\',color=\'#277da1\')\n    axes[1].bar(x+.18,subset.at_least_3,width=.36,label=\'3+ radars\',color=\'#f9844a\')\n    axes[1].set(xticks=x, xticklabels=[f\'{r.margin_m:g} m / {r.min_points} pt\' for r in subset.itertuples()], ylabel=\'Object-keyframes\',title=\'Matching sensitivity at 100 ms\')\n    axes[1].legend()\n    fig.tight_layout()\n    fig.savefig(out / \'coverage.png\', dpi=160)\n    plt.close(fig)\n    if best:\n        fig, ax = plt.subplots(figsize=(8,7))\n        ego = np.asarray(best[\'ego\'][\'translation\'])\n        for channel, (_, age, points, _) in best[\'selected\'].items():\n            if age <= 100:\n                ax.scatter(points[0]-ego[0], points[1]-ego[1],s=9,label=f\'{channel} ({age:.0f} ms)\')\n        ann = best[\'ann\']\n        yaw = Quaternion(ann[\'rotation\']).yaw_pitch_roll[0]\n        width,length,_ = ann[\'size\']\n        corners = np.array([[length/2,width/2],[length/2,-width/2],[-length/2,-width/2],[-length/2,width/2],[length/2,width/2]])\n        rot = np.array([[np.cos(yaw),-np.sin(yaw)],[np.sin(yaw),np.cos(yaw)]])\n        corners = corners @ rot.T + np.asarray(ann[\'translation\'])[:2] - ego[:2]\n        ax.plot(corners[:,0],corners[:,1],\'k-\',linewidth=2,label=\'Selected annotation (unpadded)\')\n        ax.scatter([0],[0],marker=\'*\',s=120,c=\'black\',label=\'Ego position\')\n        ax.set(xlim=(-65,65),ylim=(-65,65),xlabel=\'Global x relative to ego (m)\',ylabel=\'Global y relative to ego (m)\',title=f\'{best["scene"]}: candidate with {best["count"]} supporting radars\\nOffline geometric match; latest causal sweeps\')\n        ax.set_aspect(\'equal\')\n        ax.legend(fontsize=7,loc=\'upper left\')\n        fig.tight_layout()\n        fig.savefig(out / \'example_overlap.png\',dpi=160)\n        plt.close(fig)\n        (out / \'example_manifest.json\').write_text(json.dumps({**best[\'row\'], \'sweep_tokens\': {c:v[0][\'token\'] for c,v in best[\'selected\'].items()}},indent=2),encoding=\'utf-8\')\n\n\ndef write_report(m, summaries, candidates, out):\n    p=m[\'primary\']; n=p[\'object_keyframes\']; observed=n-p[\'sensors_0\']\n    lines=[\'# nuScenes mini radar coverage audit\',\'\',\n           \'This is a measured offline feasibility audit, not attack detection or mitigation validation.\',\'\',\n           f"Loaded {m[\'scenes\']} scenes, {m[\'keyframes\']} keyframes and {m[\'radar_files\']} radar files.",\n           f"Primary denominator: {n} vehicle/pedestrian object-keyframes within 60 m of ego; {observed} have radar support.",\n           \'\', \'| Supporting sensors | Object-keyframes |\',\'|---|---:|\']\n    lines += [f\'| {i} | {p[f"sensors_{i}"]} |\' for i in range(6)]\n    lines += [\'\',f"Two-plus support: {p[\'at_least_2\']}/{n} ({p[\'at_least_2\']/max(n,1):.2%} of eligible; {p[\'at_least_2\']/max(observed,1):.2%} of radar-observed).",\n              f"Three-plus support: {p[\'at_least_3\']}/{n} ({p[\'at_least_3\']/max(n,1):.2%} of eligible).",\'\',\n              f"Fixed-pair candidate episodes (>=3 consecutive keyframes, >=0.9 s endpoint span): {m[\'candidate_pair_episodes\']} in {m[\'candidate_pair_scenes\']} scenes.",\n              f"Fixed-triple candidate episodes under the same rule: {m[\'candidate_triple_episodes\']} in {m[\'candidate_triple_scenes\']} scenes.",\n              f"Moving objects (offline speed >=0.5 m/s): {m[\'moving_object_keyframes\']} object-keyframes; {m[\'moving_2plus\']} with 2+ support; {m[\'moving_3plus\']} with 3+ support.",\'\',\n              \'## Interpretation\',\'\',\n              \'Pair overlap supplies candidate sequences for two-stream corruption. A fixed triple may supply two corrupted streams plus a third observing stream; it does not guarantee the third is independent or sufficient to identify truth.\',\n              \'A few candidate episodes justify a pilot only. Sustained support across independent scenes and held-out data is needed before a publication-scale claim. Inspect candidates visually and rerun stricter matching before selecting attacks.\',\'\',\n              \'## Sensitivity\',\'\', \'| Configuration | Eligible | 2+ | 3+ |\',\'|---|---:|---:|---:|\']\n    lines += [f"| {s[\'id\']} | {s[\'object_keyframes\']} | {s[\'at_least_2\']} | {s[\'at_least_3\']} |" for s in summaries]\n    lines += [\'\', \'## Method and limitations\',\'\'] + [\'- \'+s for s in m[\'limitations\']]\n    lines += [\'- Primary: latest sweep at or before annotation timestamp, age <=100 ms; >=1 valid return in an oriented BEV box padded by 0.5 m per side.\',\n              \'- A point in multiple annotated boxes is excluded from every box. No stacked multisweep cloud is treated as simultaneous evidence.\',\n              \'- Annotations and annotation-derived velocity are confined to this offline audit. They must never become online detector inputs.\',\n              \'- Official default radar validity filters are explicit in summary.json; unfiltered radar results are not claimed.\',\n              \'- Ground-truth category/range restriction determines the audit denominator; lack of support is not proof of lack of field of view.\',\n              \'\', \'Source: https://github.com/nutonomy/nuscenes-devkit and https://www.nuscenes.org/nuscenes\',\n              \'Reproduce with scripts/audit_radar_overlap.py. CSVs, hashes, package versions and parameters accompany this report.\']\n    (out / \'REPORT.md\').write_text(\'\\n\'.join(lines)+\'\\n\',encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--dataroot\',default=\'data/nuscenes\')\n    p.add_argument(\'--output\',default=\'outputs/mini_audit\')\n    args=p.parse_args()\n    run(args.dataroot,args.output)\n', 'scripts/infer_replay.py': '"""Run inference using a canonical input file, without annotations or labels."""\nimport argparse\nimport json\nfrom pathlib import Path\nfrom replay_core import replay\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'input\')\n    parser.add_argument(\'output\')\n    parser.add_argument(\'--protected\', action=\'store_true\')\n    args = parser.parse_args()\n    data = json.loads(Path(args.input).read_text())\n    result = replay(data[\'events\'], data[\'frame_times\'], args.protected)\n    Path(args.output).write_text(json.dumps(result, allow_nan=False), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/prepare_nuscenes.py': '"""Download official mini archive; safely extract radar, metadata and base maps.\n\nThe compressed archive contains all modalities (~4.17 GB). Selective extraction\nreduces disk use, not download size. No credentials or unofficial mirrors.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport tarfile\nimport time\nimport urllib.request\nfrom pathlib import Path, PurePosixPath\n\nURL = \'https://www.nuscenes.org/data/v1.0-mini.tgz\'\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef download(url, target):\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with urllib.request.urlopen(urllib.request.Request(url, method=\'HEAD\'), timeout=60) as r:\n        size = int(r.headers[\'Content-Length\'])\n        etag = r.headers.get(\'ETag\')\n    if target.exists() and target.stat().st_size == size:\n        return {\'url\': url, \'bytes\': size, \'etag\': etag, \'sha256\': sha256(target)}\n    part = target.with_suffix(target.suffix + \'.part\')\n    meta = part.with_suffix(part.suffix + \'.json\')\n    identity = {\'url\': url, \'bytes\': size, \'etag\': etag}\n    if part.exists() and (not meta.exists() or json.loads(meta.read_text()) != identity):\n        raise RuntimeError(\'Partial download identity mismatch; move it aside before retrying.\')\n    meta.write_text(json.dumps(identity), encoding=\'utf-8\')\n    for attempt in range(5):\n        start = part.stat().st_size if part.exists() else 0\n        if start > size:\n            raise RuntimeError(\'Partial file larger than source.\')\n        if start == size:\n            break\n        req = urllib.request.Request(url, headers={\'Range\': f\'bytes={start}-\'} if start else {})\n        try:\n            with urllib.request.urlopen(req, timeout=90) as response:\n                if start and (response.status != 206 or not response.headers.get(\'Content-Range\', \'\').startswith(f\'bytes {start}-\')):\n                    raise RuntimeError(\'Server did not honor resume range; refusing to append.\')\n                with part.open(\'ab\' if start else \'wb\') as out:\n                    last = time.monotonic()\n                    while chunk := response.read(4 * 1024 * 1024):\n                        out.write(chunk)\n                        start += len(chunk)\n                        if time.monotonic() - last > 15:\n                            print(f\'Download {start / size:.1%} ({start / 1e9:.2f}/{size / 1e9:.2f} GB)\', flush=True)\n                            last = time.monotonic()\n            if part.stat().st_size == size:\n                break\n        except (OSError, TimeoutError) as exc:\n            print(f\'Download attempt {attempt+1}: {exc}; resumable partial retained.\', flush=True)\n            if attempt == 4:\n                raise\n    if part.stat().st_size != size:\n        raise RuntimeError(\'Incomplete archive.\')\n    part.replace(target)\n    return {**identity, \'sha256\': sha256(target)}\n\n\ndef selected_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in name or \':\' in name:\n        raise ValueError(f\'Unsafe archive path: {name}\')\n    return (p.parts and p.parts[0] in (\'v1.0-mini\', \'maps\')) or (\n        len(p.parts) >= 3 and p.parts[0] in (\'samples\', \'sweeps\') and p.parts[1].startswith(\'RADAR_\'))\n\n\ndef extract(archive, root):\n    root = Path(root).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    files = []\n    # Streaming mode avoids keeping a multi-GB archive index in memory.\n    with tarfile.open(archive, \'r|gz\') as tar:\n        for member in tar:\n            if not selected_member(member.name) or not member.isfile():\n                continue\n            dest = root.joinpath(*PurePosixPath(member.name).parts)\n            if not dest.resolve().is_relative_to(root):\n                raise ValueError(\'Extraction escapes destination.\')\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            source = tar.extractfile(member)\n            with source, dest.open(\'wb\') as out:\n                shutil.copyfileobj(source, out)\n            files.append({\'path\': dest.relative_to(root).as_posix(), \'bytes\': member.size, \'sha256\': sha256(dest)})\n            if len(files) % 5000 == 0:\n                print(f\'Extracted {len(files)} selected files\', flush=True)\n    if not (root / \'v1.0-mini/scene.json\').exists():\n        raise RuntimeError(\'Missing mini metadata after extraction.\')\n    return files\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--archive\', default=\'downloads/v1.0-mini.tgz\')\n    parser.add_argument(\'--dataroot\', default=\'data/nuscenes\')\n    parser.add_argument(\'--url\', default=URL)\n    args = parser.parse_args()\n    root = Path(args.dataroot)\n    marker = root / \'preparation_manifest.json\'\n    if marker.exists():\n        manifest = json.loads(marker.read_text(encoding=\'utf-8\'))\n        if all((root / f[\'path\']).exists() and (root / f[\'path\']).stat().st_size == f[\'bytes\'] for f in manifest[\'files\']):\n            print(\'Prepared dataset present (size check); use audit hashes for provenance.\')\n            return\n        raise RuntimeError(\'Prepared dataset incomplete; move preparation_manifest.json aside to re-extract.\')\n    print(\'Downloading official nuScenes mini; dataset use remains subject to nuScenes terms.\', flush=True)\n    provenance = download(args.url, args.archive)\n    files = extract(args.archive, root)\n    marker.write_text(json.dumps({\'archive\': provenance, \'extraction\': \'radar + metadata + base maps\', \'files\': files}, indent=2), encoding=\'utf-8\')\n    print(f\'Prepared {len(files)} files at {root.resolve()}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/render_replay.py': '"""Render saved experimental output as an offline, self-contained replay display."""\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\n\n\ndef render(out, results, summaries, metrics, references, attack, provenance):\n    out = Path(out)\n    names = [\'clean_baseline\', \'attacked_baseline\', \'attacked_protected\']\n    colors = [\'#38bdf8\', \'#fb7185\', \'#34d399\']\n    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)\n    for name, color in zip([*names, \'clean_protected\'], [*colors, \'#a78bfa\']):\n        rows = metrics[name]\n        axes[0].plot([r[\'time_s\'] for r in rows], [r[\'gospa\'] for r in rows], label=name.replace(\'_\', \' \'), color=color)\n        axes[1].plot([r[\'time_s\'] for r in rows], [r[\'false_tracks\'] + r[\'missed_targets\'] for r in rows], color=color)\n    for ax in axes:\n        ax.axvspan(attack[\'start_s\'], attack[\'end_s\'], color=\'orange\', alpha=.15)\n        ax.grid(alpha=.2)\n    axes[0].set_ylabel(\'GOSPA (p=2, c=10m, alpha=2)\')\n    axes[1].set_ylabel(\'False + missed hypotheses\')\n    axes[1].set_xlabel(\'Scene time (s); shaded interval = injected attack\')\n    axes[0].legend()\n    fig.suptitle(f\'{provenance["scene"]} — measured training-scene pilot, heuristic baseline\')\n    fig.tight_layout()\n    fig.savefig(out / \'tracking_comparison.png\', dpi=150)\n    plt.close(fig)\n    report = [\'# Causal radar replay pilot\', \'\', f\'Scene: `{provenance["scene"]}`. Split: **train / development only**. Run: `{provenance["run_id"]}`.\', \'\',\n              \'Measured report-level simulation on recorded radar data. No learned model or RF validation.\', \'\',\n              \'| Branch | Mean GOSPA ↓ | Matched RMSE (m) | False track-frames | Missed target-frames |\',\n              \'|---|---:|---:|---:|---:|\']\n    for name, s in summaries.items():\n        rmse = s[\'matched_position_rmse_m\']\n        report.append(f\'| {name} | {s["mean_gospa"]:.3f} | {rmse:.3f} | {s["false_track_frames"]} | {s["missed_target_frames"]} |\' if rmse is not None else f\'| {name} | {s["mean_gospa"]:.3f} | n/a | {s["false_track_frames"]} | {s["missed_target_frames"]} |\')\n    report += [\'\', \'RMSE includes only matches within 4m and must be read with false/missed counts. GOSPA includes all emitted hypotheses and eligible annotations; reflection centroids are not object centers.\', \'\',\n               \'## Detector measurements\', \'\', \'| Branch | TP | FP | FN | Precision | Recall | First event delay (s) |\', \'|---|---:|---:|---:|---:|---:|---:|\']\n    fmt = lambda x: \'n/a\' if x is None else f\'{x:.3f}\'\n    for name, s in summaries.items():\n        d = s[\'detection\']\n        report.append(f\'| {name} | {d["tp"]} | {d["fp"]} | {d["fn"]} | {fmt(d["precision"])} | {fmt(d["recall"])} | {fmt(d["event_delay_s"])} |\')\n    report += [\'\', \'The baseline branches run the detector in monitor-only mode. Protected branches reject flagged updates and inflate covariance using heuristic temporal trust. No detection means a null delay, never zero.\', \'\',\n               \'False-alert counts are report decisions, not independent attack events. All rows come from one development scene and one injection; there is no held-out performance claim.\', \'\',\n               \'Compare each attacked branch with its own clean branch. Protection also changes clean tracking, so a lower protected GOSPA alone is not evidence of successful attack mitigation.\', \'\',\n               \'## Limits and next work\', \'\'] + [\'- \' + text for text in provenance[\'limitations\']]\n    report += [\'\', \'Next: inspect episodes and clutter, calibrate frontend/residual thresholds on training/validation recordings, add benign stresses and additional attack families, then compare GRU and temporal graph detection. See HANDOFF.md.\', \'\',\n               \'Open `demo.html` for playback, scrub controls, reference overlays, and clean/attacked/protected comparisons. All display annotations are offline overlays; inference cannot access them.\']\n    (out / \'REPORT.md\').write_text(\'\\n\'.join(report), encoding=\'utf-8\')\n    raw = {}\n    for name in (\'clean\', \'attacked\'):\n        events = json.loads((out / f\'inputs/{name}.json\').read_text())[\'events\']\n        latest, cursor, snapshots = {}, 0, []\n        for ref in references:\n            while cursor < len(events) and events[cursor][\'time_s\'] <= ref[\'time_s\']:\n                e = events[cursor]\n                latest[e[\'sensor\']] = e\n                cursor += 1\n            snapshots.append([r[\'position\'] for e in latest.values() if ref[\'time_s\'] - e[\'time_s\'] <= .2 for r in e[\'reports\']])\n        raw[name] = snapshots\n    compact = {name: [dict(fused=f[\'fused\'], tracks=[dict(state=t[\'state\'], trust=t[\'trust\'], sensor=t[\'sensor\']) for t in f[\'tracks\']])\n                      for f in results[name][\'frames\']] for name in names}\n    payload = dict(scene=provenance[\'scene\'], run_id=provenance[\'run_id\'], attack=attack, summaries=summaries, references=references,\n                   results=compact, raw=raw, metrics=metrics, sources=provenance[\'sources\'])\n    html = TEMPLATE.replace(\'__DATA__\', json.dumps(payload, allow_nan=False).replace(\'</\', \'<\\\\/\'))\n    (out / \'demo.html\').write_text(html, encoding=\'utf-8\')\n\n\nTEMPLATE = r\'\'\'<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Radar Trust Lab — Recorded Replay</title>\n<style>\n:root{color-scheme:dark;font-family:system-ui,sans-serif;background:#09111e;color:#e2e8f0}body{margin:0;padding:28px;max-width:1600px;margin:auto}h1{font-size:30px;margin:8px 0}h2{font-size:17px;margin:0 0 8px}p{color:#aebbd0;line-height:1.6}.eyebrow{color:#38bdf8;font-size:12px;letter-spacing:.15em;text-transform:uppercase}.badge{display:inline-block;background:#20314a;padding:5px 10px;border-radius:20px;font-size:12px;margin-right:6px}.controls{display:flex;gap:18px;align-items:center;flex-wrap:wrap;padding:18px;background:#121f32;border-radius:12px;margin:20px 0}button,select{background:#1d3652;color:white;border:1px solid #466382;border-radius:6px;padding:9px;cursor:pointer}input[type=range]{flex:1;min-width:160px;accent-color:#38bdf8}.grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:16px}.card{background:#111e30;border:1px solid #24354d;border-radius:12px;padding:16px;min-width:0}canvas{width:100%;height:auto;background:#0a1525;border-radius:8px}.legend{display:flex;gap:18px;flex-wrap:wrap;font-size:13px;margin:12px 0}.note{border-left:3px solid #fbbf24;padding:10px 16px;background:#29251a;color:#f3ddb0}table{width:100%;border-collapse:collapse;font-size:13px}th,td{text-align:left;padding:10px;border-bottom:1px solid #24354d}.scroll{overflow-x:auto}.small{font-size:12px;color:#9fb0c7}.metric{font-size:13px;margin-top:12px;line-height:1.7}.bottom{display:grid;grid-template-columns:1.3fr 1fr;gap:16px;margin-top:18px}a{color:#7dd3fc}summary{cursor:pointer}#state{font-weight:600;color:#fbbf24}@media(max-width:1000px){.grid,.bottom{grid-template-columns:1fr}body{padding:16px}}\n</style></head><body>\n<div class="eyebrow">Radar Trust Lab / Research prototype</div><h1>When radar reports disagree</h1>\n<p>Replay recorded driving data, introduce a shared displacement into two radar streams, and inspect a causal detector’s response.</p>\n<span class="badge" id="scene"></span><span class="badge">Training-scene pilot</span><span class="badge">Heuristic residual detector · no trained GNN</span>\n<div class="controls"><button id="play">▶ Play</button><button id="reset">Restart</button><input id="scrub" type="range" min="0" value="0" aria-label="Replay frame"><span id="time"></span><select id="speed" aria-label="Playback speed"><option value="1">1× speed</option><option value="2">2× speed</option><option value="0.5">0.5× speed</option></select><label><input type="checkbox" id="focus" checked> Focus target</label><label><input type="checkbox" id="truth" checked> Offline reference</label></div>\n<div id="state" aria-live="polite"></div><div class="legend"><span style="color:#94a3b8">· Radar report centroids</span><span style="color:#e2e8f0">□ Offline annotated boxes</span><span style="color:#fbbf24">◇ Selected target</span><span>○ Fused estimates + 1σ position ellipses</span></div>\n<div class="grid"><section class="card"><h2 style="color:#38bdf8">1 / Clean baseline</h2><div class="small">Recorded input · monitoring only</div><canvas id="clean" width="520" height="420"></canvas><div class="metric" id="m0"></div></section><section class="card"><h2 style="color:#fb7185">2 / Attacked baseline</h2><div class="small">Modified input · monitoring only</div><canvas id="attacked" width="520" height="420"></canvas><div class="metric" id="m1"></div></section><section class="card"><h2 style="color:#34d399">3 / Protected replay</h2><div class="small">Same modified input · reject + coast + trust inflation</div><canvas id="protected" width="520" height="420"></canvas><div class="metric" id="m2"></div></section></div>\n<div class="bottom"><section class="card"><h2>Tracking error through the replay</h2><canvas id="chart" width="850" height="240"></canvas><p class="small">GOSPA includes position error, false tracks and missed targets. Lower is better. Shading marks the injection interval. The white cursor follows playback.</p></section><section class="card"><h2>What the system knows</h2><p id="attack"></p><p class="note">Agreement does not prove truth. Two coherently altered streams can evade this detector. New or unmatched reports remain unknown; missing evidence is not proof of an attack.</p><p class="small">Trust is a heuristic score, not a calibrated probability. Reference boxes, target selection and attack status are added only by this offline display. Focus mode also uses the offline target position.</p></section></div>\n<section class="card" style="margin-top:18px"><h2>Measured results — including the clean-data cost</h2><div class="scroll"><table><thead><tr><th>Branch</th><th>Mean GOSPA ↓</th><th>Matched RMSE (m)</th><th>False / missed track-frames</th><th>Detection precision / recall</th><th>False alert reports/min</th></tr></thead><tbody id="metrics"></tbody></table></div><p class="small">One development scene and one injected event. These are not held-out accuracy results. RMSE uses 4m gated matches; read it alongside false/missed counts. All branches use the same frontend and fusion rule.</p></section>\n<details style="margin-top:18px"><summary>Method, reproducibility and presentation notes</summary><p>Five causal radar streams → position clustering → local Kalman trackers → innovation/CUSUM monitoring → optional rejection and trust inflation → equal-weight covariance intersection. A stale track coasts for at most 0.8s; no hidden reconstruction of deleted information.</p><p>Press Play, pause inside the highlighted attack interval, compare the same target across the three panels, then read the measured results. An unchanged or worse protected result is a valid observed outcome, not a display error.</p><p>Coordinate axes are east/north offsets in a translated global frame. Ellipses depict modeled 1σ position uncertainty, not a certified coverage guarantee. Background clutter and sparse returns affect the metrics.</p><p><a href="REPORT.md">Measured report</a> · <a href="summary.json">Metrics JSON</a> · <a href="provenance.json">Source hashes and parameters</a> · <a href="tracking_comparison.png">Exportable comparison figure</a></p><pre id="hashes" style="white-space:pre-wrap;font-size:11px"></pre></details>\n<script>\nconst D=__DATA__;\nconst names=[\'clean_baseline\',\'attacked_baseline\',\'attacked_protected\'], colors=[\'#38bdf8\',\'#fb7185\',\'#34d399\'];\nconst canvases=[\'clean\',\'attacked\',\'protected\'].map(id=>document.getElementById(id));\nconst $=id=>document.getElementById(id);let index=0,playing=false,elapsed=0,last=0;\nconst fmt=x=>x===null||x===undefined?\'n/a\':Number(x).toFixed(2);\n$(\'scene\').textContent=D.scene+\' · run \'+D.run_id;$(\'scrub\').max=D.references.length-1;\n$(\'attack\').textContent=`Injection: ${D.attack.sensors.join(\' + \')}. Shared offset [${D.attack.displacement_m.join(\', \')}] m, ${fmt(D.attack.start_s)}–${fmt(D.attack.end_s)}s. Actual altered reports: ${Object.values(D.attack.actual_corrupted_report_counts).join(\' + \')}.`;\n$(\'hashes\').textContent=JSON.stringify(D.sources,null,2);\nfor(const [name,s] of Object.entries(D.summaries)){const tr=document.createElement(\'tr\');for(const value of [name.replaceAll(\'_\',\' \'),fmt(s.mean_gospa),fmt(s.matched_position_rmse_m),`${s.false_track_frames} / ${s.missed_target_frames}`,`${fmt(s.detection.precision)} / ${fmt(s.detection.recall)}`,fmt(s.detection.false_alert_reports_per_minute)]){const td=document.createElement(\'td\');td.textContent=value;tr.appendChild(td)}$(\'metrics\').appendChild(tr)}\nfunction bounds(ref){const target=ref.objects.find(o=>o.instance===D.attack.target_instance);const center=$(\'focus\').checked&&target?target.position:ref.ego;const span=$(\'focus\').checked?35:130;return {center,span}}\nfunction drawPanel(canvas,k,ref){const ctx=canvas.getContext(\'2d\'),w=canvas.width,h=canvas.height,b=bounds(ref),scale=Math.min(w,h)/b.span;\nconst project=p=>[w/2+(p[0]-b.center[0])*scale,h/2-(p[1]-b.center[1])*scale];ctx.clearRect(0,0,w,h);ctx.strokeStyle=\'#1d3048\';ctx.lineWidth=1;\nfor(let a=-1000;a<=1000;a+=5){const x=project([a,0])[0],y=project([0,a])[1];if(x>=0&&x<=w){ctx.beginPath();ctx.moveTo(x,0);ctx.lineTo(x,h);ctx.stroke()}if(y>=0&&y<=h){ctx.beginPath();ctx.moveTo(0,y);ctx.lineTo(w,y);ctx.stroke()}}\nctx.fillStyle=\'#73849b\';for(const p of D.raw[k===0?\'clean\':\'attacked\'][index]){const [x,y]=project(p);ctx.beginPath();ctx.arc(x,y,2,0,Math.PI*2);ctx.fill()}\nif($(\'truth\').checked)for(const o of ref.objects){if(!o.eligible)continue;const [x,y]=project(o.position);ctx.save();ctx.translate(x,y);ctx.rotate(-o.yaw);ctx.strokeStyle=o.instance===D.attack.target_instance?\'#fbbf24\':\'#cbd5e180\';ctx.lineWidth=o.instance===D.attack.target_instance?2:1;ctx.strokeRect(-o.size[1]*scale/2,-o.size[0]*scale/2,o.size[1]*scale,o.size[0]*scale);ctx.restore()}\nfor(const r of D.results[names[k]][index].fused){const [x,y]=project(r.state),p=r.covariance,aa=p[0][0],bb=p[0][1],cc=p[1][1],disc=Math.sqrt((aa-cc)**2+4*bb*bb),l1=Math.max(.01,(aa+cc+disc)/2),l2=Math.max(.01,(aa+cc-disc)/2);ctx.strokeStyle=colors[k]+\'70\';ctx.beginPath();ctx.ellipse(x,y,Math.sqrt(l1)*scale,Math.sqrt(l2)*scale,-.5*Math.atan2(2*bb,aa-cc),0,2*Math.PI);ctx.stroke();ctx.strokeStyle=colors[k];ctx.beginPath();ctx.arc(x,y,4,0,2*Math.PI);ctx.stroke();ctx.fillStyle=colors[k];ctx.font=\'10px system-ui\';ctx.fillText(`${r.support}R`,x+6,y-5)}\nctx.fillStyle=\'#aebbd0\';ctx.font=\'11px system-ui\';ctx.fillText(\'North ↑   East →   grid: 5 m\',12,h-12);\nconst m=D.metrics[names[k]][index],tracks=D.results[names[k]][index].tracks,trust=tracks.length?tracks.reduce((s,t)=>s+t.trust,0)/tracks.length:null;\n$(\'m\'+k).textContent=`GOSPA ${fmt(m.gospa)} · false ${m.false_tracks} · missed ${m.missed_targets} | mean local-track trust ${fmt(trust)}`;\n}\nfunction chart(){const c=$(\'chart\'),ctx=c.getContext(\'2d\'),w=c.width,h=c.height,left=45,top=15,bottom=h-28,end=D.references.at(-1).time_s,max=Math.max(1,...names.flatMap(n=>D.metrics[n].map(r=>r.gospa)))*1.1;const x=t=>left+t/end*(w-left-15),y=v=>bottom-v/max*(bottom-top);ctx.clearRect(0,0,w,h);ctx.fillStyle=\'#fbbf2420\';ctx.fillRect(x(D.attack.start_s),top,x(D.attack.end_s)-x(D.attack.start_s),bottom-top);ctx.font=\'11px system-ui\';ctx.fillStyle=\'#9fb0c7\';for(let j=0;j<=4;j++){const v=j*max/4;ctx.fillText(v.toFixed(0),5,y(v)+4);ctx.strokeStyle=\'#24354d\';ctx.beginPath();ctx.moveTo(left,y(v));ctx.lineTo(w,y(v));ctx.stroke()}\nnames.forEach((n,k)=>{ctx.strokeStyle=colors[k];ctx.lineWidth=2;ctx.beginPath();D.metrics[n].forEach((r,j)=>j?ctx.lineTo(x(r.time_s),y(r.gospa)):ctx.moveTo(x(r.time_s),y(r.gospa)));ctx.stroke()});ctx.strokeStyle=\'white\';ctx.beginPath();ctx.moveTo(x(D.references[index].time_s),top);ctx.lineTo(x(D.references[index].time_s),bottom);ctx.stroke();ctx.fillText(\'0s\',left,h-6);ctx.fillText(end.toFixed(1)+\'s\',w-40,h-6)}\nfunction render(){const ref=D.references[index];$(\'scrub\').value=index;$(\'time\').textContent=fmt(ref.time_s)+\' s\';$(\'state\').textContent=ref.time_s>=D.attack.start_s&&ref.time_s<=D.attack.end_s?\'INJECTION WINDOW · offline experiment label\':ref.time_s<D.attack.start_s?\'BEFORE INJECTION · establish causal history\':\'AFTER INJECTION · inspect recovery and stale tracks\';canvases.forEach((c,k)=>drawPanel(c,k,ref));chart()}\n$(\'play\').onclick=()=>{if(index===D.references.length-1){index=0;elapsed=0}playing=!playing;$(\'play\').textContent=playing?\'❚❚ Pause\':\'▶ Play\';render()};$(\'reset\').onclick=()=>{index=0;elapsed=0;playing=false;$(\'play\').textContent=\'▶ Play\';render()};$(\'scrub\').oninput=e=>{index=Number(e.target.value);elapsed=0;render()};$(\'focus\').onchange=render;$(\'truth\').onchange=render;\nfunction tick(now){if(playing&&last){elapsed+=(now-last)/1000*Number($(\'speed\').value);while(index<D.references.length-1&&elapsed>=D.references[index+1].time_s-D.references[index].time_s){elapsed-=D.references[index+1].time_s-D.references[index].time_s;index++;render()}if(index===D.references.length-1){playing=false;$(\'play\').textContent=\'▶ Play\'}}last=now;requestAnimationFrame(tick)}render();requestAnimationFrame(tick);\n</script></body></html>\'\'\'\n', 'scripts/replay_core.py': '"""Annotation-free causal tracking and conservative fusion for report replay.\n\nThis module intentionally has no nuScenes, ground-truth, or attack-label API.\nCovariances and thresholds are declared pilot heuristics, not calibrated values.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nimport time\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\n\n@dataclass(frozen=True)\nclass Config:\n    acceleration_std: float = 3.0\n    association_gate_m: float = 12.0\n    association_nis: float = 36.0\n    alert_nis: float = 9.21\n    cusum_drift: float = 2.5\n    cusum_threshold: float = 10.0\n    max_coast_s: float = 0.8\n    confirmation_hits: int = 3\n    fusion_gate_m: float = 3.0\n    max_sensor_age_s: float = 0.2\n\n\n@dataclass\nclass Track:\n    id: str\n    x: np.ndarray\n    p: np.ndarray\n    timestamp: float\n    last_update: float\n    hits: int = 1\n    cusum: float = 0.0\n    trust: float = 0.5\n    suspect: bool = False\n\n\ndef predict(x, p, dt, acceleration_std):\n    if dt < -1e-8:\n        raise ValueError(\'Cannot predict backwards in time\')\n    dt = max(0.0, dt)\n    f = np.eye(4)\n    f[0, 2] = f[1, 3] = dt\n    g = np.array([[dt * dt / 2, 0], [0, dt * dt / 2], [dt, 0], [0, dt]])\n    return f @ x, f @ p @ f.T + acceleration_std ** 2 * g @ g.T\n\n\ndef innovation(track, report):\n    z, r = np.asarray(report[\'position\'], float), np.asarray(report[\'covariance\'], float)\n    residual = z - track.x[:2]\n    s = track.p[:2, :2] + r\n    return residual, s, float(residual @ np.linalg.solve(s, residual))\n\n\nclass LocalTracker:\n    def __init__(self, sensor, protected=False, config=None):\n        self.sensor = sensor\n        self.protected = protected\n        self.config = config or Config()\n        self.tracks = []\n        self.counter = 0\n        self.last_timestamp = -np.inf\n\n    def update(self, event):\n        t = event[\'time_s\']\n        if t <= self.last_timestamp:\n            raise ValueError(\'Each sensor sweep must be consumed once in increasing time order\')\n        self.last_timestamp = t\n        cfg = self.config\n        reports = event[\'reports\']\n        for report in reports:\n            z, r = np.asarray(report[\'position\']), np.asarray(report[\'covariance\'])\n            if z.shape != (2,) or r.shape != (2, 2) or not np.isfinite(z).all() or not np.isfinite(r).all():\n                raise ValueError(\'Malformed report\')\n            if not np.allclose(r, r.T) or np.linalg.eigvalsh(r).min() <= 0:\n                raise ValueError(\'Report covariance must be positive definite\')\n        self.tracks = [a for a in self.tracks if t - a.last_update <= cfg.max_coast_s]\n        for a in self.tracks:\n            a.x, a.p = predict(a.x, a.p, t - a.timestamp, cfg.acceleration_std)\n            a.timestamp = t\n        costs = np.full((len(self.tracks), len(reports)), 1e9)\n        for i, a in enumerate(self.tracks):\n            for j, r in enumerate(reports):\n                residual, _, nis = innovation(a, r)\n                if np.linalg.norm(residual) <= cfg.association_gate_m and nis <= cfg.association_nis:\n                    costs[i, j] = nis\n        matches = {}\n        if costs.size:\n            ii, jj = linear_sum_assignment(costs)\n            matches = {int(j): int(i) for i, j in zip(ii, jj) if costs[i, j] < 1e9}\n        decisions = []\n        for j, r in enumerate(reports):\n            if j not in matches:\n                # Unmatched reports remain visible and monitored; no unsupported\n                # accusation or trust reduction merely because a report is new.\n                self.counter += 1\n                a = Track(f\'{self.sensor}:{self.counter}\', np.r_[r[\'position\'], 0., 0.],\n                          np.diag([4., 4., 100., 100.]), t, t)\n                a.p[:2, :2] = np.asarray(r[\'covariance\'])\n                self.tracks.append(a)\n                decisions.append(dict(report_id=r[\'id\'], track_id=a.id, nis=None,\n                                      score=0., alert=False, status=\'unknown_new\', trust=a.trust))\n                continue\n            a = self.tracks[matches[j]]\n            residual, s, nis = innovation(a, r)\n            # Two-dimensional innovations have expectation 2 only under a\n            # correctly specified model. These pilot settings need calibration.\n            a.cusum = max(0., a.cusum + nis - cfg.cusum_drift)\n            established = a.hits >= cfg.confirmation_hits\n            alert = established and (nis > cfg.alert_nis or a.cusum > cfg.cusum_threshold)\n            score = max(nis / cfg.alert_nis, a.cusum / cfg.cusum_threshold) if established else 0.\n            quality = 0. if alert else 1.\n            alpha = 0.5 if alert else 0.08\n            a.trust = (1 - alpha) * a.trust + alpha * quality\n            a.suspect = alert\n            if not (self.protected and alert):\n                k = np.linalg.solve(s, a.p[:, :2].T).T\n                a.x = a.x + k @ residual\n                kh = np.zeros((4, 4))\n                kh[:, :2] = k\n                identity = np.eye(4) - kh\n                a.p = identity @ a.p @ identity.T + k @ np.asarray(r[\'covariance\']) @ k.T\n                a.last_update = t\n                a.hits += 1\n            decisions.append(dict(report_id=r[\'id\'], track_id=a.id, nis=nis, score=score,\n                                  alert=bool(alert), status=\'suspected_corruption\' if alert else\n                                  (\'accepted\' if established else \'tentative\'), trust=a.trust))\n        return decisions\n\n    def snapshot(self, t):\n        result = []\n        for a in self.tracks:\n            age = t - a.last_update\n            if age > self.config.max_coast_s or a.hits < self.config.confirmation_hits:\n                continue\n            x, p = predict(a.x, a.p, t - a.timestamp, self.config.acceleration_std)\n            result.append(dict(id=a.id, sensor=self.sensor, state=x.tolist(), covariance=p.tolist(),\n                               age_s=age, trust=a.trust, suspect=a.suspect,\n                               status=\'coasting\' if age > self.config.max_sensor_age_s else \'observed\'))\n        return result\n\n\ndef fuse_tracks(tracks, config=None, protected=False):\n    """Complete-link spatial grouping, one report per sensor, equal-weight CI.\n\n    Equal weights are fixed to avoid tuning on evaluation labels. No majority or\n    independence guarantee is made. Unassociated hypotheses remain in the output.\n    """\n    cfg = config or Config()\n    groups = []\n    for track in sorted(tracks, key=lambda r: r[\'id\']):\n        target = None\n        for group in groups:\n            if track[\'sensor\'] in {r[\'sensor\'] for r in group}:\n                continue\n            if all(np.linalg.norm(np.array(track[\'state\'][:2]) - r[\'state\'][:2]) <= cfg.fusion_gate_m for r in group):\n                target = group\n                break\n        if target is None:\n            groups.append([track])\n        else:\n            target.append(track)\n    fused = []\n    for group in groups:\n        infos = [np.linalg.inv(np.asarray(r[\'covariance\']) / max(r[\'trust\'], .1)\n                               if protected else np.asarray(r[\'covariance\'])) for r in group]\n        precision = sum(infos) / len(infos)\n        p = np.linalg.inv(precision)\n        x = p @ (sum(info @ r[\'state\'] for info, r in zip(infos, group)) / len(group))\n        observed = sum(r[\'age_s\'] <= cfg.max_sensor_age_s and not r[\'suspect\'] for r in group)\n        fused.append(dict(state=x.tolist(), covariance=p.tolist(), members=[r[\'id\'] for r in group],\n                          support=len(group), observed_support=observed,\n                          status=\'degraded\' if observed == 0 else \'single_source\' if observed == 1\n                          else \'multi_source_unverified\'))\n    return fused\n\n\ndef replay(events, frame_times, protected=False, config=None):\n    """Only events and timestamps are accepted, never annotations or labels."""\n    cfg = config or Config()\n    if any(events[i][\'time_s\'] > events[i + 1][\'time_s\'] for i in range(len(events) - 1)):\n        raise ValueError(\'Replay events must be globally time ordered\')\n    if any(frame_times[i] >= frame_times[i + 1] for i in range(len(frame_times) - 1)):\n        raise ValueError(\'Frame times must increase\')\n    trackers, frames, decisions, latency = {}, [], [], []\n    cursor = 0\n    for t in frame_times:\n        while cursor < len(events) and events[cursor][\'time_s\'] <= t:\n            e = events[cursor]\n            start = time.perf_counter()\n            tracker = trackers.setdefault(e[\'sensor\'], LocalTracker(e[\'sensor\'], protected, cfg))\n            for row in tracker.update(e):\n                decisions.append(dict(time_s=e[\'time_s\'], sensor=e[\'sensor\'], **row))\n            latency.append((time.perf_counter() - start) * 1000)\n            cursor += 1\n        tracks = [r for tracker in trackers.values() for r in tracker.snapshot(t)]\n        frames.append(dict(time_s=t, tracks=tracks, fused=fuse_tracks(tracks, cfg, protected)))\n    return dict(frames=frames, decisions=decisions,\n                timing=dict(event_count=cursor, local_update_ms_p50=float(np.percentile(latency, 50)) if latency else 0.,\n                            local_update_ms_p95=float(np.percentile(latency, 95)) if latency else 0.))\n', 'scripts/run_replay.py': '"""Prepare and evaluate a real-data report-level attack pilot; write a portable demo.\n\nAnnotations are used only by the offline injector and evaluator. The online API\nis scripts/replay_core.py and can be run from exported inputs without nuScenes.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import asdict\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport platform\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom nuscenes.nuscenes import NuScenes\nfrom nuscenes.utils.data_classes import RadarPointCloud\nfrom pyquaternion import Quaternion\nfrom scipy.optimize import linear_sum_assignment\nfrom sklearn.cluster import DBSCAN\n\nfrom replay_core import Config, replay\n\nROOT = Path(__file__).resolve().parents[1]\nSENSORS = [\'RADAR_FRONT\', \'RADAR_FRONT_LEFT\', \'RADAR_FRONT_RIGHT\', \'RADAR_BACK_LEFT\', \'RADAR_BACK_RIGHT\']\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef freeze_splits(nusc, path):\n    """Group whole recordings before features, attack generation or fitting."""\n    scenes = sorted(nusc.scene, key=lambda s: s[\'name\'])\n    groups = sorted({s[\'log_token\'] for s in scenes}, key=lambda s: hashlib.sha256((\'20261109:\' + s).encode()).hexdigest())\n    n = len(groups)\n    if n < 3:\n        raise ValueError(\'At least three recordings are required for grouped pilot splits\')\n    ntest = max(1, n // 5)\n    split = {g: (\'test\' if i < ntest else \'validation\' if i < 2 * ntest else \'train\') for i, g in enumerate(groups)}\n    manifest = dict(schema_version=1, dataset=\'v1.0-mini\', purpose=\'pilot only; not a publication benchmark\',\n                    grouping=\'nuScenes log_token; nearby recordings may still be correlated\', seed=20261109,\n                    scenes=[dict(scene=s[\'name\'], scene_token=s[\'token\'], log_token=s[\'log_token\'], split=split[s[\'log_token\']]) for s in scenes])\n    path = Path(path)\n    if path.exists():\n        existing = json.loads(path.read_text())\n        if existing != manifest:\n            raise ValueError(\'Existing split manifest differs; refusing to silently replace it\')\n    else:\n        write_json(path, manifest)\n    return manifest\n\n\ndef extract(nusc, scene):\n    samples = []\n    token = scene[\'first_sample_token\']\n    while token:\n        sample = nusc.get(\'sample\', token)\n        samples.append(sample)\n        token = sample[\'next\']\n    start_us = samples[0][\'timestamp\']\n    first_sd = nusc.get(\'sample_data\', samples[0][\'data\'][\'RADAR_FRONT\'])\n    origin = np.asarray(nusc.get(\'ego_pose\', first_sd[\'ego_pose_token\'])[\'translation\'])\n    last_us = samples[-1][\'timestamp\']\n    records = [sd for sd in nusc.sample_data if sd[\'channel\'] in SENSORS\n               and nusc.get(\'sample\', sd[\'sample_token\'])[\'scene_token\'] == scene[\'token\']\n               and sd[\'timestamp\'] <= last_us]\n    records.sort(key=lambda sd: (sd[\'timestamp\'], sd[\'channel\']))\n    events, sources = [], []\n    for sd in records:\n        path = Path(nusc.dataroot) / sd[\'filename\']\n        points = RadarPointCloud.from_file(str(path), invalid_states=[0], dynprop_states=list(range(7)), ambig_states=[3]).points\n        valid = np.isfinite(points[:3]).all(axis=0) & (np.linalg.norm(points[:2], axis=0) <= 60.)\n        points = points[:, valid]\n        cs = nusc.get(\'calibrated_sensor\', sd[\'calibrated_sensor_token\'])\n        pose = nusc.get(\'ego_pose\', sd[\'ego_pose_token\'])\n        xyz = Quaternion(cs[\'rotation\']).rotation_matrix @ points[:3] + np.asarray(cs[\'translation\'])[:, None]\n        xyz = Quaternion(pose[\'rotation\']).rotation_matrix @ xyz + np.asarray(pose[\'translation\'])[:, None] - origin[:, None]\n        xy = xyz[:2].T\n        reports = []\n        if len(xy):\n            labels = DBSCAN(eps=2., min_samples=1).fit_predict(xy)\n            for label in sorted(set(labels)):\n                mask = labels == label\n                cluster = xy[mask]\n                # Centroids are radar reflection centers, not annotated box centers.\n                scatter = np.cov(cluster.T) / len(cluster) if len(cluster) > 1 else np.zeros((2, 2))\n                r = scatter + np.eye(2) * 1.5 ** 2\n                rcs = points[5, mask]\n                reports.append(dict(id=f\'{sd["token"]}:{label}\', position=cluster.mean(axis=0).tolist(),\n                                    covariance=r.tolist(), point_count=len(cluster),\n                                    rcs_mean=float(np.mean(rcs[np.isfinite(rcs)])) if np.isfinite(rcs).any() else None))\n        events.append(dict(sensor=sd[\'channel\'], time_s=(sd[\'timestamp\'] - start_us) / 1e6,\n                           acquisition_timestamp_us=sd[\'timestamp\'], reception_timestamp_us=None,\n                           sweep_token=sd[\'token\'], reports=reports))\n        sources.append(dict(file=sd[\'filename\'], sha256=sha(path)))\n    references = []\n    for sample in samples:\n        sd = nusc.get(\'sample_data\', sample[\'data\'][\'RADAR_FRONT\'])\n        ego = np.asarray(nusc.get(\'ego_pose\', sd[\'ego_pose_token\'])[\'translation\'])\n        objects = []\n        for token in sample[\'anns\']:\n            a = nusc.get(\'sample_annotation\', token)\n            category = a[\'category_name\']\n            eligible = ((category.startswith(\'vehicle.\') and category != \'vehicle.bicycle_rack\')\n                        or category.startswith(\'human.pedestrian.\')) and np.linalg.norm(np.asarray(a[\'translation\'])[:2] - ego[:2]) <= 60\n            objects.append(dict(instance=a[\'instance_token\'], position=(np.asarray(a[\'translation\']) - origin)[:2].tolist(),\n                                size=a[\'size\'], yaw=Quaternion(a[\'rotation\']).yaw_pitch_roll[0],\n                                category=category, eligible=bool(eligible)))\n        references.append(dict(time_s=(sample[\'timestamp\'] - start_us) / 1e6, sample_token=sample[\'token\'],\n                               ego=(ego - origin)[:2].tolist(), objects=objects))\n    return events, references, dict(origin_global=origin.tolist(), start_timestamp_us=start_us, radar_files=sources)\n\n\ndef inside(position, obj, padding=.5):\n    delta = np.asarray(position) - obj[\'position\']\n    c, s = np.cos(obj[\'yaw\']), np.sin(obj[\'yaw\'])\n    local = np.array([[c, s], [-s, c]]) @ delta\n    return abs(local[0]) <= obj[\'size\'][1] / 2 + padding and abs(local[1]) <= obj[\'size\'][0] / 2 + padding\n\n\ndef inject(events, references, attack):\n    """Offline annotation-assisted target selection, never an inference feature.\n\n    Use the latest annotation box for a transparent pilot association proxy.\n    Shared shifts at equal times implement a coordinated displacement process.\n    """\n    altered = json.loads(json.dumps(events))\n    labels = []\n    times = [r[\'time_s\'] for r in references]\n    for event in altered:\n        t = event[\'time_s\']\n        idx = int(np.searchsorted(times, t, side=\'right\')) - 1\n        objects = references[idx][\'objects\'] if idx >= 0 else []\n        for report in event[\'reports\']:\n            matches = [o for o in objects if inside(report[\'position\'], o)]\n            active = (attack[\'start_s\'] <= t <= attack[\'end_s\'] and event[\'sensor\'] in attack[\'sensors\']\n                      and len(matches) == 1 and matches[0][\'instance\'] == attack[\'target_instance\'])\n            if active:\n                scale = min(1., (t - attack[\'start_s\']) / attack[\'ramp_s\']) if attack[\'ramp_s\'] > 0 else 1.\n                shift = np.asarray(attack[\'displacement_m\']) * scale\n                report[\'position\'] = (np.asarray(report[\'position\']) + shift).tolist()\n                active = bool(np.linalg.norm(shift) > 0)\n            labels.append(dict(report_id=report[\'id\'], time_s=t, sensor=event[\'sensor\'], attacked=bool(active)))\n    return altered, labels\n\n\ndef frame_metrics(estimates, truth, cutoff=10., match_gate=4.):\n    """GOSPA p=2, alpha=2; independent 4m matching for RMSE and counts."""\n    a, b = np.asarray(estimates).reshape(-1, 2), np.asarray(truth).reshape(-1, 2)\n    n, m = len(a), len(b)\n    distances = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)\n    if n and m:\n        ii, jj = linear_sum_assignment(np.minimum(distances ** 2, cutoff ** 2))\n        gospa2 = float(np.minimum(distances[ii, jj] ** 2, cutoff ** 2).sum()) + cutoff ** 2 / 2 * abs(n - m)\n        # Maximize number of gated matches first, then minimize distance.\n        cost = np.where(distances <= match_gate, distances / max(n, m, 1), 1e6)\n        ii, jj = linear_sum_assignment(cost)\n        errors = distances[ii, jj][distances[ii, jj] <= match_gate]\n    else:\n        gospa2, errors = cutoff ** 2 / 2 * (n + m), np.array([])\n    return dict(gospa=float(np.sqrt(gospa2)), matched=len(errors), false_tracks=n - len(errors),\n                missed_targets=m - len(errors), squared_error_sum=float((errors ** 2).sum()))\n\n\ndef evaluate(result, references, labels, attack):\n    rows = []\n    for frame, ref in zip(result[\'frames\'], references):\n        # Count all emitted hypotheses, including clutter, against all eligible GT.\n        truth = [o[\'position\'] for o in ref[\'objects\'] if o[\'eligible\']]\n        row = frame_metrics([r[\'state\'][:2] for r in frame[\'fused\']], truth)\n        row.update(time_s=frame[\'time_s\'])\n        rows.append(row)\n    label_map = {r[\'report_id\']: r[\'attacked\'] for r in labels}\n    scored = [dict(**r, attacked=label_map[r[\'report_id\']]) for r in result[\'decisions\']]\n    tp = sum(r[\'alert\'] and r[\'attacked\'] for r in scored)\n    fp = sum(r[\'alert\'] and not r[\'attacked\'] for r in scored)\n    fn = sum(not r[\'alert\'] and r[\'attacked\'] for r in scored)\n    tn = sum(not r[\'alert\'] and not r[\'attacked\'] for r in scored)\n    matched = sum(r[\'matched\'] for r in rows)\n    actual = [r[\'time_s\'] for r in scored if r[\'attacked\']]\n    hits = [r[\'time_s\'] for r in scored if r[\'attacked\'] and r[\'alert\']]\n    clean_minutes = (references[-1][\'time_s\'] - references[0][\'time_s\']) / 60\n    summary = dict(mean_gospa=float(np.mean([r[\'gospa\'] for r in rows])),\n                   matched_position_rmse_m=float(np.sqrt(sum(r[\'squared_error_sum\'] for r in rows) / matched)) if matched else None,\n                   matched_object_frames=matched, false_track_frames=sum(r[\'false_tracks\'] for r in rows),\n                   missed_target_frames=sum(r[\'missed_targets\'] for r in rows),\n                   detection=dict(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),\n                                  precision=tp / (tp + fp) if tp + fp else None,\n                                  recall=tp / (tp + fn) if tp + fn else None,\n                                  f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,\n                                  attacked_reports=len(actual), event_detected=bool(hits) if actual else None,\n                                  event_delay_s=min(hits) - min(actual) if hits else None,\n                                  # Per-report alerts, not deduplicated attack events.\n                                  false_alert_reports_per_minute=fp / clean_minutes if clean_minutes else None),\n                   timing=result[\'timing\'])\n    return summary, rows, scored\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--dataroot\', default=str(ROOT / \'data/nuscenes\'))\n    parser.add_argument(\'--output\', default=str(ROOT / \'outputs/replay_pilot\'))\n    parser.add_argument(\'--scene\', help=\'Training scene only; default is longest audited training episode\')\n    parser.add_argument(\'--displacement\', type=float, default=4.)\n    parser.add_argument(\'--ramp\', type=float, default=0.)\n    args = parser.parse_args()\n    if not np.isfinite(args.displacement) or not np.isfinite(args.ramp) or args.ramp < 0 or args.displacement == 0:\n        parser.error(\'Displacement must be finite and nonzero; ramp must be finite and nonnegative\')\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=True)\n    started = time.perf_counter()\n    nusc = NuScenes(version=\'v1.0-mini\', dataroot=args.dataroot, verbose=False)\n    manifest_path = ROOT / \'data_manifests/mini_splits.json\'\n    splits = freeze_splits(nusc, manifest_path)\n    training = {s[\'scene\'] for s in splits[\'scenes\'] if s[\'split\'] == \'train\'}\n    candidates_path = ROOT / \'outputs/mini_audit/candidate_episodes.csv\'\n    candidates = pd.read_csv(candidates_path)\n    candidates = candidates[(candidates.sensor_count == 2) & candidates.scene.isin(training) & (candidates.duration_s >= 4)]\n    if args.scene:\n        candidates = candidates[candidates.scene == args.scene]\n    if candidates.empty:\n        raise ValueError(\'No >=4s training episode found. Run the overlap audit first; do not use held-out scenes for the demo.\')\n    episode = candidates.sort_values([\'duration_s\', \'scene\'], ascending=[False, True]).iloc[0]\n    scene = next(s for s in nusc.scene if s[\'name\'] == episode.scene)\n    print(f\'Preparing {scene["name"]}: training-only pilot, target {episode.instance_token}\', flush=True)\n    events, references, provenance = extract(nusc, scene)\n    time_lookup = {r[\'sample_token\']: r[\'time_s\'] for r in references}\n    start = time_lookup[episode.start_sample] + 2.\n    end = min(start + 6., time_lookup[episode.end_sample])\n    attack = dict(type=\'coordinated_position_displacement\', injection_boundary=\'clustered position reports before tracking\',\n                  target_instance=episode.instance_token, sensors=episode.sensors.split(\'|\'),\n                  start_s=start, end_s=end, displacement_m=[args.displacement, 0.], ramp_s=args.ramp,\n                  target_selection=\'exclusive membership in latest offline annotation box +0.5m; association proxy\',\n                  rf_validated=False)\n    attacked_events, labels = inject(events, references, attack)\n    counts = {s: sum(r[\'attacked\'] and r[\'sensor\'] == s for r in labels) for s in attack[\'sensors\']}\n    if not all(counts.values()):\n        raise RuntimeError(f\'No actual corruption in one of the selected streams: {counts}\')\n    attack[\'actual_corrupted_report_counts\'] = counts\n    frame_times = [r[\'time_s\'] for r in references]\n    # These files are sufficient for inference. No annotation/label fields inside.\n    write_json(out / \'inputs/clean.json\', dict(events=events, frame_times=frame_times))\n    write_json(out / \'inputs/attacked.json\', dict(events=attacked_events, frame_times=frame_times))\n    write_json(out / \'evaluation/reference.json\', references)\n    write_json(out / \'evaluation/attack.json\', attack)\n    write_json(out / \'evaluation/labels.json\', labels)\n    cfg = Config()\n    results, summaries, metric_rows = {}, {}, {}\n    clean_labels = [dict(r, attacked=False) for r in labels]\n    for name, input_events, protected, input_labels in [\n        (\'clean_baseline\', events, False, clean_labels), (\'clean_protected\', events, True, clean_labels),\n        (\'attacked_baseline\', attacked_events, False, labels), (\'attacked_protected\', attacked_events, True, labels)]:\n        print(f\'Running {name} ...\', flush=True)\n        result = replay(input_events, frame_times, protected, cfg)\n        summary, rows, scored = evaluate(result, references, input_labels, attack)\n        write_json(out / f\'runs/{name}.json\', result)\n        pd.DataFrame(rows).to_csv(out / f\'{name}_tracking.csv\', index=False)\n        pd.DataFrame(scored).to_csv(out / f\'{name}_decisions.csv\', index=False)\n        results[name], summaries[name], metric_rows[name] = result, summary, rows\n    source_paths = [ROOT / \'scripts\' / name for name in (\'replay_core.py\', \'run_replay.py\', \'render_replay.py\', \'infer_replay.py\', \'verify_replay.py\')]\n    provenance.update(scene=scene[\'name\'], scene_token=scene[\'token\'], split=\'train\',\n                      manifest_sha256=sha(manifest_path), config=asdict(cfg),\n                      preprocessing=dict(range_m=60, dbscan_eps_m=2, min_samples=1, centroid_std_floor_m=1.5),\n                      sources={p.name: sha(p) for p in source_paths},\n                      metadata_sha256={p.name: sha(p) for p in sorted((Path(args.dataroot) / \'v1.0-mini\').glob(\'*.json\'))},\n                      inputs_sha256={name: sha(out / f\'inputs/{name}.json\') for name in (\'clean\', \'attacked\')},\n                      evaluation_sha256={name: sha(out / f\'evaluation/{name}.json\') for name in (\'attack\', \'labels\', \'reference\')},\n                      candidate_manifest_sha256=sha(candidates_path), requirements_sha256=sha(ROOT / \'requirements.txt\'),\n                      environment=dict(python=platform.python_version(), platform=platform.platform(),\n                                       packages={p: importlib.metadata.version(p) for p in [\'numpy\', \'scipy\', \'scikit-learn\', \'nuscenes-devkit\', \'pandas\']}),\n                      elapsed_before_render_s=time.perf_counter() - started,\n                      limitations=[\'Pilot heuristic thresholds; no validation calibration or trained model.\',\n                                   \'No hardware/RF validation. Report-level injection after clustering.\',\n                                   \'Reflection centroids do not equal object centers; full scene clutter retained.\',\n                                   \'Three-radar recovery not established. Shared pose errors unmodeled.\',\n                                   \'New/unmatched reports are unknown; sustained corruption may initialize new tracks.\',\n                                   \'No visibility-aware suppression detector or persistent fused IDs yet.\',\n                                   \'Local-update timing excludes decoding, feature extraction, fusion and display.\'])\n    identity = {key: provenance[key] for key in (\'sources\', \'inputs_sha256\', \'evaluation_sha256\', \'config\', \'manifest_sha256\')}\n    provenance[\'run_id\'] = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]\n    write_json(out / \'provenance.json\', provenance)\n    write_json(out / \'summary.json\', summaries)\n    from render_replay import render\n    render(out, results, summaries, metric_rows, references, attack, provenance)\n    print(f\'Finished. Open {out / "demo.html"}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/verify_replay.py': '"""Re-run protected inference in a directory containing no labels or annotations."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', default=\'outputs/replay_pilot\')\n    args = parser.parse_args()\n    out = Path(args.output).resolve()\n    root = Path(__file__).resolve().parents[1]\n    checks = out / \'checks\'\n    checks.mkdir(exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix=\'annotation_free_\', dir=checks) as temporary:\n        work = Path(temporary)\n        for name in (\'replay_core.py\', \'infer_replay.py\'):\n            shutil.copy2(root / \'scripts\' / name, work / name)\n        shutil.copy2(out / \'inputs/attacked.json\', work / \'input.json\')\n        subprocess.run([sys.executable, str(work / \'infer_replay.py\'), \'input.json\', \'result.json\', \'--protected\'],\n                       cwd=work, check=True)\n        actual = json.loads((work / \'result.json\').read_text())\n        expected = json.loads((out / \'runs/attacked_protected.json\').read_text())\n        # Wall-clock profiling is explicitly nondeterministic; scientific outputs are not.\n        actual.pop(\'timing\')\n        expected.pop(\'timing\')\n        if actual != expected:\n            raise AssertionError(\'Isolated inference differs from saved experiment\')\n        result = dict(passed=True, check=\'isolated annotation-free inference reproduces frames and decisions\',\n                      compared_fields=[\'frames\', \'decisions\'], excluded_fields=[\'timing\'],\n                      input_sha256=hashlib.sha256((work / \'input.json\').read_bytes()).hexdigest(),\n                      frame_count=len(actual[\'frames\']), decision_count=len(actual[\'decisions\']))\n    (checks / \'inference_isolation.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'tests/test_audit.py': "import importlib.util\nfrom pathlib import Path\nimport numpy as np\nfrom pyquaternion import Quaternion\n\nspec=importlib.util.spec_from_file_location('audit',Path(__file__).parents[1]/'scripts/audit_radar_overlap.py')\naudit=importlib.util.module_from_spec(spec)\nspec.loader.exec_module(audit)\n\n\ndef test_causal_selection_never_uses_future():\n    assert audit.latest_causal([90,110],100,20)==0\n    assert audit.latest_causal([110],100,20) is None\n    assert audit.latest_causal([70],100,20) is None\n    assert audit.latest_causal([100],100,0)==0\n\n\ndef test_sensor_to_global_rotation_translation():\n    calibration={'rotation':Quaternion(axis=[0,0,1],angle=np.pi/2).elements,'translation':[1,0,0]}\n    pose={'rotation':[1,0,0,0],'translation':[10,20,0]}\n    result=audit.transform_points(np.array([[2.],[0.],[0.]]),calibration,pose)\n    np.testing.assert_allclose(result[:,0],[11,22,0],atol=1e-10)\n\n\ndef test_box_dimensions_rotation_and_ambiguity():\n    box={'translation':[0,0,0],'rotation':Quaternion(axis=[0,0,1],angle=np.pi/2).elements,'size':[2,4,2]}\n    pts=np.array([[0,1.5,0],[1.5,0,2.2],[0,0,0]])\n    hit=audit.box_membership(pts,[box],0)\n    assert hit.tolist()==[[True,False,False]]\n    assert audit.box_membership(pts,[box],.5).tolist()==[[True,True,True]]\n    counts,ambiguous=audit.exclusive_counts(np.array([[True,True,False],[True,False,True]]))\n    assert counts.tolist()==[1,1] and ambiguous==1\n\n\ndef test_episodes_require_fixed_pair_and_consecutive_frames():\n    def row(i,sensors):\n        return dict(scene='s',instance_token='a',sensors=sensors,frame_index=i,timestamp_us=i*500000,sample_token=str(i),category='vehicle.car')\n    rows=[row(0,['A','B']),row(1,['A','C']),row(2,['A','B']),row(3,['A','B'])]\n    result=audit.sustained_episodes(rows,2)\n    assert max(r['keyframes'] for r in result)==2\n    assert result[0]['sensors']=='A|B'\n\n\ndef test_archive_path_traversal_rejected():\n    import pytest\n    prep_spec=importlib.util.spec_from_file_location('prep',Path(__file__).parents[1]/'scripts/prepare_nuscenes.py')\n    prep=importlib.util.module_from_spec(prep_spec)\n    prep_spec.loader.exec_module(prep)\n    with pytest.raises(ValueError):\n        prep.selected_member('../escape')\n    assert prep.selected_member('samples/RADAR_FRONT/example.pcd')\n    assert not prep.selected_member('samples/CAM_FRONT/example.jpg')\n", 'tests/test_replay.py': '"""Known-answer checks for causality, uncertainty, labels and metric accounting."""\nimport json\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nimport pytest\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \'scripts\'))\nfrom replay_core import Config, LocalTracker, fuse_tracks, predict, replay\nfrom run_replay import frame_metrics, freeze_splits, inject\n\n\ndef event(t, x=0., sensor=\'A\'):\n    return dict(time_s=t, sensor=sensor, reports=[dict(id=f\'{sensor}:{t}\', position=[x, 0.], covariance=[[1., 0.], [0., 1.]])])\n\n\ndef test_predict_uncertainty_and_backwards_time():\n    x, p = predict(np.array([0., 0., 2., 0.]), np.eye(4), .5, 3.)\n    assert x[0] == 1.\n    assert p[0, 0] > 1.\n    assert np.linalg.eigvalsh(p).min() > 0\n    with pytest.raises(ValueError):\n        predict(x, p, -.1, 3.)\n\n\ndef test_prefix_invariance_and_no_future_reports():\n    events = [event(i / 10, i / 10) for i in range(20)]\n    prefix = replay(events[:10], [.3, .9])\n    full = replay(events, [.3, .9, 1.9])\n    assert prefix[\'frames\'] == full[\'frames\'][:2]\n    assert all(r[\'time_s\'] <= .9 for r in prefix[\'decisions\'])\n\n\ndef test_duplicate_and_unordered_sweeps_rejected():\n    with pytest.raises(ValueError):\n        replay([event(0.), event(0.)], [0.])\n    with pytest.raises(ValueError):\n        replay([event(1.), event(0.)], [1.])\n\n\ndef test_protection_rejects_jump_and_stale_track_expires():\n    events = [event(i / 10) for i in range(20)] + [event(2., 5.)]\n    raw = replay(events, [1.9, 2.])\n    protected = replay(events, [1.9, 2., 3.], True)\n    assert protected[\'decisions\'][-1][\'alert\']\n    assert protected[\'frames\'][1][\'tracks\'][0][\'state\'][0] == 0.\n    assert raw[\'frames\'][1][\'tracks\'][0][\'state\'][0] > 0.\n    assert not protected[\'frames\'][-1][\'tracks\']\n\n\ndef test_unmatched_report_remains_unknown_and_visible():\n    r = replay([event(0), event(.1, 100)], [.1])\n    assert r[\'decisions\'][-1][\'status\'] == \'unknown_new\'\n    assert not r[\'decisions\'][-1][\'alert\']\n\n\ndef test_invalid_covariance_rejected():\n    e = event(0)\n    e[\'reports\'][0][\'covariance\'] = [[-1, 0], [0, 1]]\n    with pytest.raises(ValueError):\n        replay([e], [0])\n\n\ndef test_no_observation_does_not_imply_attack():\n    tracker = LocalTracker(\'A\')\n    tracker.update(event(0))\n    trust = tracker.tracks[0].trust\n    assert tracker.update(dict(time_s=.1, sensor=\'A\', reports=[])) == []\n    assert tracker.tracks[0].trust == trust\n\n\ndef test_ci_does_not_double_identical_evidence_precision():\n    def track(sensor):\n        return dict(id=sensor, sensor=sensor, state=[0, 0, 1, 0], covariance=np.eye(4).tolist(), age_s=0, suspect=False, trust=1)\n    fused = fuse_tracks([track(\'A\'), track(\'B\')])\n    assert len(fused) == 1\n    np.testing.assert_allclose(fused[0][\'covariance\'], np.eye(4))\n    assert len(fuse_tracks([track(\'A\'), track(\'A\')])) == 2\n\n\ndef test_metrics_count_unmatched_and_empty_cases():\n    assert frame_metrics([], [])[\'gospa\'] == 0\n    assert frame_metrics([], [[0, 0]])[\'missed_targets\'] == 1\n    assert frame_metrics([[100, 0]], [[0, 0]])[\'gospa\'] == 10\n    r = frame_metrics([[3, 0], [100, 0]], [[0, 0]])\n    assert r[\'squared_error_sum\'] == 9\n    assert r[\'false_tracks\'] == 1 and r[\'missed_targets\'] == 0\n    assert r[\'gospa\'] == pytest.approx(np.sqrt(59))\n\n\ndef test_injector_is_repeatable_does_not_mutate_or_leak_labels():\n    events = [event(0), event(.1), event(.1, sensor=\'B\'), event(.1, sensor=\'C\')]\n    ref = [dict(time_s=0, objects=[dict(instance=\'target\', position=[0, 0], yaw=0, size=[2, 4, 2])])]\n    attack = dict(start_s=.05, end_s=1, sensors=[\'A\', \'B\'], target_instance=\'target\', displacement_m=[4, 0], ramp_s=0)\n    altered, labels = inject(events, ref, attack)\n    assert altered == inject(events, ref, attack)[0]\n    assert events[1][\'reports\'][0][\'position\'] == [0, 0]\n    assert altered[1][\'reports\'][0][\'position\'] == [4, 0]\n    assert [r[\'attacked\'] for r in labels] == [False, True, True, False]\n    assert \'attacked\' not in json.dumps(altered) and \'target\' not in json.dumps(altered)\n    assert replay(altered, [0, .1])[\'frames\'] == replay(json.loads(json.dumps(altered)), [0, .1])[\'frames\']\n\n\ndef test_split_manifest_groups_recordings_and_refuses_changes(tmp_path):\n    class Dataset:\n        scene = [dict(name=f\'s{i}\', token=f\'t{i}\', log_token=f\'l{i // 2}\') for i in range(10)]\n    path = tmp_path / \'splits.json\'\n    manifest = freeze_splits(Dataset(), path)\n    for log in {r[\'log_token\'] for r in manifest[\'scenes\']}:\n        assert len({r[\'split\'] for r in manifest[\'scenes\'] if r[\'log_token\'] == log}) == 1\n    assert freeze_splits(Dataset(), path) == manifest\n    path.write_text(\'{}\')\n    with pytest.raises(ValueError):\n        freeze_splits(Dataset(), path)\n'}
for relative, source in FILES.items():
    destination = PROJECT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Project sources installed:', sorted(FILES))


In [ ]:
# Isolate Python 3.12 from Colab's kernel; uv does not require ensurepip.
def run_logged(command, cwd=PROJECT):
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError(f'Command failed: {command}')
run_logged([sys.executable, '-m', 'pip', 'install', 'uv'])
PYTHON = PROJECT / '.venv' / ('Scripts/python.exe' if sys.platform == 'win32' else 'bin/python')
if not PYTHON.exists():
    run_logged([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(PROJECT / '.venv')])
run_logged([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(PYTHON), '-r', str(PROJECT / 'requirements.txt')])
run_logged([sys.executable, '-m', 'uv', 'pip', 'check', '--python', str(PYTHON)])
run_logged([str(PYTHON), '-m', 'pytest', '-q', '--basetemp=outputs/test_tmp', 'tests'])


## Download and prepare
The first run downloads ~4.17 GB. The script resumes interrupted transfers and records
the archive SHA-256. Extraction retains no camera or LiDAR payload. Repeat runs reuse
the prepared subset after a size check. Files live in the temporary runtime; download
the results before disconnecting. Google Drive is not mounted or granted broad access.


In [ ]:
run_logged([str(PYTHON), '-u', 'scripts/prepare_nuscenes.py'])


## Run the offline coverage audit
Uses the latest sweep **at or before** each annotation timestamp. Tests 50/100/200 ms,
0/0.5 m BEV padding, and 1/2 valid points per sensor. No multisweep accumulation.
Overlapping-box points are excluded. The fixed-pair/triple episode metric requires
at least 3 consecutive keyframes and a 0.9 s endpoint span.

**Important:** annotation-derived velocity can use future annotations, solely to align
boxes for this offline audit. It must never be exposed to the later detector. Radar
points in a box are a geometric coverage proxy, not proven point-to-object identities.


In [ ]:
run_logged([str(PYTHON), '-u', 'scripts/audit_radar_overlap.py'])


## Build the replay experiment

Freeze recording-grouped splits, select an audited **training** episode, export
annotation-free reports, and run four paired branches. Injection uses offline
annotations; inference has no access to them. Heuristic parameters are fixed in source.


In [ ]:
run_logged([str(PYTHON), '-u', 'scripts/run_replay.py'])
run_logged([str(PYTHON), '-u', 'scripts/verify_replay.py'])


In [ ]:
from IPython.display import display, Markdown, Image
RESULTS = PROJECT / 'outputs/replay_pilot'
display(Markdown((RESULTS / 'REPORT.md').read_text()))
display(Image(filename=str(RESULTS / 'tracking_comparison.png')))


## Present and preserve the experiment

Download the ZIP, extract it, and open **demo.html** in a desktop browser. It has
embedded replay data, playback, scrubbing, reference overlays, and measured results.
No server or internet connection is required for the HTML itself.

For your presentation: explain the clean replay, pause in the attack window,
compare the protected branch, and discuss false alerts and observability limits.
The current baseline can fail; this is explicitly shown, not hidden.


In [ ]:
archive = shutil.make_archive(str(PROJECT / 'replay_pilot_results'), 'zip', RESULTS)
print('Download this archive before the runtime ends:', archive)
print('Desktop demo:', RESULTS / 'demo.html')
# from google.colab import files
# files.download(archive)
